In [9]:
!pip install playwright

In [10]:
!playwright install chromium

In [11]:
# ── STEP 1: One-time login ───────────────────────────────────────────────────
# Browser opens → fill email & password → window closes by itself when done.
# Re-run only if Step 2 says "Session expired".

import asyncio, sys, threading
from playwright.async_api import async_playwright

REPORT_URL  = "https://app.powerbi.com/groups/me/reports/36c0dd62-e78a-4f8a-a7b9-8ef80ffcda46/28a6e6bc3a83d0b03f92?experience=power-bi"
PROFILE_DIR = "./powerbi_profile"

LOGIN_SELECTORS = (
    'input[type="email"], input[type="password"], '
    'input[name="loginfmt"], input[name="passwd"], '
    '#i0116, #i0118, '
    'input[placeholder*="mail" i], input[placeholder*="mot" i], '
    '.ms-TextField input'
)

async def setup_login():
    async with async_playwright() as p:
        context = await p.chromium.launch_persistent_context(
            user_data_dir=PROFILE_DIR,
            headless=False,
            viewport={"width": 1920, "height": 1080},
        )
        page = await context.new_page()
        await page.goto(REPORT_URL)
        try:
            await page.wait_for_load_state("load", timeout=15000)
        except Exception:
            pass
        await page.wait_for_timeout(5000)

        print("⏳  Loading login page...")
        form_appeared = False
        try:
            await page.wait_for_selector(LOGIN_SELECTORS, state="visible", timeout=15000)
            form_appeared = True
        except Exception:
            pass

        if not form_appeared:
            already_in = (
                "app.powerbi.com" in page.url and
                not await page.evaluate(f"() => !!document.querySelector(`{LOGIN_SELECTORS}`)")
            )
            if already_in:
                print("✅  Already authenticated — session saved.")
                await context.close()
                return

        print("=" * 55)
        print("🔐  Fill in your email and password in the")
        print("    browser window that just opened.")
        print("=" * 55)

        for tick in range(120):
            await page.wait_for_timeout(3000)
            try:
                url      = page.url
                has_form = await page.evaluate(
                    f"() => !!document.querySelector(`{LOGIN_SELECTORS}`)"
                )
                if "app.powerbi.com" in url and not has_form:
                    await page.wait_for_timeout(2000)
                    has_form2 = await page.evaluate(
                        f"() => !!document.querySelector(`{LOGIN_SELECTORS}`)"
                    )
                    if not has_form2:
                        break
            except Exception:
                pass
            if tick > 0 and tick % 10 == 0:
                print(f"   Still waiting... ({tick * 3}s)")
        else:
            print("⏱  6-minute timeout. Re-run this cell and try again.")
            await context.close()
            return

        await page.wait_for_timeout(2000)
        print("✅  Logged in! Session saved — run Step 2 now.")
        await context.close()

def run():
    loop = asyncio.ProactorEventLoop() if sys.platform == "win32" else asyncio.new_event_loop()
    asyncio.set_event_loop(loop)
    try:
        loop.run_until_complete(setup_login())
    finally:
        loop.close()

thread = threading.Thread(target=run)
thread.start()
thread.join()

Exception in thread Thread-17 (run):
Traceback (most recent call last):
  File "c:\Users\USER\miniconda3\Lib\threading.py", line 1073, in _bootstrap_inner
    self.run()
  File "c:\Users\USER\miniconda3\Lib\threading.py", line 1010, in run
    self._target(*self._args, **self._kwargs)
  File "C:\Users\USER\AppData\Local\Temp\ipykernel_17368\22318391.py", line 88, in run
  File "c:\Users\USER\miniconda3\Lib\asyncio\base_events.py", line 687, in run_until_complete
    return future.result()
           ^^^^^^^^^^^^^^^
  File "C:\Users\USER\AppData\Local\Temp\ipykernel_17368\22318391.py", line 21, in setup_login
  File "c:\Users\USER\miniconda3\Lib\site-packages\playwright\async_api\_generated.py", line 16606, in launch_persistent_context
    await self._impl_obj.launch_persistent_context(
  File "c:\Users\USER\miniconda3\Lib\site-packages\playwright\_impl\_browser_type.py", line 166, in launch_persistent_context
    result = await self._channel.send_return_as_dict(
             ^^^^^^^^^^

In [12]:
# ── STEP 2: Extract data + drill-through + send HTML email ───────────────────

import asyncio, sys, threading, smtplib, json, os, re, base64
import requests as req_lib
import pandas as pd
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText
from email.mime.image import MIMEImage
from email.mime.base import MIMEBase
from email import encoders
from datetime import datetime
from playwright.async_api import async_playwright

# ── Configuration ─────────────────────────────────────────────────────────────
REPORT_URL  = "https://app.powerbi.com/groups/me/reports/36c0dd62-e78a-4f8a-a7b9-8ef80ffcda46/28a6e6bc3a83d0b03f92?experience=power-bi"
REPORT_ID   = "36c0dd62-e78a-4f8a-a7b9-8ef80ffcda46"
PROFILE_DIR = "./powerbi_profile"
EMAIL_FROM  = "aalabouzid2002@gmail.com"
EMAIL_PASSWORD = "nwsj xmwh jvxz rspv"
EMAIL_TO    = ["ala-eddine.bouzid@esprit.tn"]
API_BASE    = "https://api.powerbi.com/v1.0/myorg"
XLSX_PATH   = "dépôt2026_nettoyé.xlsx"
LOGO_PATH   = "logo_laposte.jpg"

_REGIONS = [
    "ARIANA","BEJA","BEN AROUS","BIZERTE","GABES","GAFSA","JENDOUBA",
    "KAIROUAN","KASSERINE","KEBILI","KEF","MAHDIA","MANOUBA","MEDENINE",
    "MONASTIR","NABEUL","SFAX","SIDI BOUZID","SILIANA","SOUSSE",
    "TATAOUINE","TOZEUR","TUNIS","ZAGHOUAN",
]
MANUAL_SLICER_MAP = {r: "Region Depot" for r in _REGIONS}

_COMPUTED_COLS = {
    frozenset({"Agences", "Centres de distribution", "Bureaux"}): (
        "Categorie_Bureau_Dernier_E_nle",
        lambda v: "Agences" if "agence" in str(v).lower()
                  else "Centres de distribution" if "centre" in str(v).lower()
                  else "Bureaux"
    ),
}

_BLANK_TOKENS = {"(blank)", "(vide)", "(empty)", "(null)", ""}

_NATIONAL_DIMS       = {"Bureau depot", "Region dernier E"}
_NATIONAL_EXTRA_COLS = ["poids", "Dernier E", "CA"]

C_NAVY   = "#0B2A6F"
C_YELLOW = "#F4C20D"
C_BG     = "#F7F9FC"
C_LIGHT  = "#EEF4FF"

_schema_cache = {}

# ── Logo helper ───────────────────────────────────────────────────────────────

def get_logo_b64():
    if os.path.exists(LOGO_PATH):
        with open(LOGO_PATH, "rb") as f:
            data = f.read()
        ext  = os.path.splitext(LOGO_PATH)[1].lower().lstrip(".")
        mime = "jpeg" if ext in ("jpg", "jpeg") else ext
        return f"data:image/{mime};base64," + base64.b64encode(data).decode()
    return None
def pbi_get(token, path):
    return req_lib.get(f"{API_BASE}{path}",
                       headers={"Authorization": f"Bearer {token}"}).json()

def get_dataset_id(token):
    for path in [f"/reports/{REPORT_ID}", f"/groups/me/reports/{REPORT_ID}"]:
        data = pbi_get(token, path)
        if "datasetId" in data:
            print(f"  ↳ Dataset ID: {data['datasetId']}")
            return data["datasetId"]
    return None

def run_dax(token, dataset_id, dax, silent=False):
    resp = req_lib.post(
        f"{API_BASE}/datasets/{dataset_id}/executeQueries",
        headers={"Authorization": f"Bearer {token}", "Content-Type": "application/json"},
        json={"queries": [{"query": dax}], "serializerSettings": {"includeNulls": True}}
    )
    data = resp.json()
    if "error" in data:
        if not silent:
            print(f"  ❌ DAX error: {data.get('error', {}).get('code', '')}")
        return None
    try:
        return data["results"][0]["tables"][0].get("rows", [])
    except (KeyError, IndexError):
        return []

# ── Dynamic schema discovery ──────────────────────────────────────────────────

def find_pbi_table(token, dataset_id):
    cache = _schema_cache.setdefault(dataset_id, {})
    if "tbl" in cache:
        return cache["tbl"]
    rows = run_dax(token, dataset_id,
                   "EVALUATE SELECTCOLUMNS(INFO.TABLES(), \"n\", [Name])", silent=True)
    if rows:
        skip = {"dateautotable", "localdatetable"}
        candidates = [r.get("[n]") or r.get("n") or "" for r in rows]
        candidates = [n for n in candidates
                      if n and not n.startswith("$") and n.lower() not in skip]
        for name in candidates:
            if run_dax(token, dataset_id,
                       f"EVALUATE SELECTCOLUMNS(TOPN(1,'{name}'),\"x\","
                       f"'{name}'[MAILITM_FID])", silent=True) is not None:
                print(f"  ↳ Power BI table auto-discovered: '{name}'")
                cache["tbl"] = name
                return name
    for name in ["export", "Export", "dépôt2026_nettoyé"]:
        if run_dax(token, dataset_id, f"EVALUATE TOPN(1,'{name}')", silent=True) is not None:
            print(f"  ↳ Power BI table (fallback): '{name}'")
            cache["tbl"] = name
            return name
    return None

def get_pbi_columns(token, dataset_id, tbl_name):
    cache = _schema_cache.setdefault(dataset_id, {})
    key   = f"cols_{tbl_name}"
    if key in cache:
        return cache[key]
    rows = run_dax(token, dataset_id,
                   f"EVALUATE SELECTCOLUMNS("
                   f"FILTER(INFO.COLUMNS(), [TableName] = \"{tbl_name}\"),"
                   f"\"c\", [ExplicitName])", silent=True)
    cols = [r.get("[c]") or r.get("c") or "" for r in (rows or [])]
    cols = [c for c in cols if c]
    cache[key] = cols
    return cols

def is_dax_filterable(token, dataset_id, tbl_name, col):
    return run_dax(token, dataset_id,
                   f"EVALUATE SELECTCOLUMNS(TOPN(1,'{tbl_name}'),\"t\","
                   f"'{tbl_name}'[{col}])", silent=True) is not None

def auto_resolve_slicer_dax(token, dataset_id, tbl_name, selected_values):
    if not (token and dataset_id and tbl_name):
        return None, None
    all_cols = get_pbi_columns(token, dataset_id, tbl_name)
    if not all_cols:
        return None, None
    SKIP_KW = ('ID','FID','NUM','DATE','TIME','AMOUNT','QTY',
               'COUNT','SUM','AVG','YEAR','MONTH','WEEK')
    candidates = [c for c in all_cols
                  if not any(k in c.upper() for k in SKIP_KW)][:20]
    real_vals = [v for v in selected_values if v.lower().strip() not in _BLANK_TOKENS]
    if not real_vals:
        return None, None
    val_list = ", ".join(f'"{v}"' for v in real_vals)
    n_vals   = len(set(real_vals))
    for col in candidates:
        try:
            rows = run_dax(token, dataset_id,
                           f"EVALUATE FILTER(VALUES('{tbl_name}'[{col}]),"
                           f"'{tbl_name}'[{col}] IN {{{val_list}}})", silent=True)
            if rows is not None and len(rows) >= n_vals:
                return col, is_dax_filterable(token, dataset_id, tbl_name, col)
        except Exception:
            continue
    return None, None

# ── xlsx helpers ──────────────────────────────────────────────────────────────

def norm(s):
    return re.sub(r'[\s_\-]+', '', str(s).lower())

def has_blank_selection(vals):
    return any(v.lower().strip() in _BLANK_TOKENS for v in vals)

def xlsx_filter(df, col, selected_vals):
    real = [v for v in selected_vals if v.lower().strip() not in _BLANK_TOKENS]
    has_blank = has_blank_selection(selected_vals)
    if has_blank and real:
        return df[df[col].str.strip().isin([""] + real)]
    if has_blank:
        return df[df[col].str.strip() == ""]
    return df[df[col].isin(real)]

def load_xlsx_df():
    df = pd.read_excel(XLSX_PATH, sheet_name="national", header=2, dtype=str).fillna("")
    _cat = lambda v: ("Agences" if "agence" in str(v).lower()
                      else "Centres de distribution" if "centre" in str(v).lower()
                      else "Bureaux")
    # Bureau dernier E = last event bureau (matches PBI: Bureau dernier E)
    bde_col = next((c for c in df.columns
                    if norm(c) in {norm("Bureau dernier E"), norm("Bureau next")}), None)
    if bde_col:
        df["Categorie_Bureau_Dernier_E_nle"] = df[bde_col].apply(_cat)
        print(f"  ↳ Categorie_Bureau_Dernier_E_nle computed from [{bde_col}]")
    else:
        print("  ⚠ Bureau dernier E column not found in xlsx")
    return df
def compute_kpis(fdf):
    kpis = {"livres": 0, "total": len(fdf), "taux": 0.0, "avg_intervalle": None}
    de  = next((c for c in fdf.columns if norm(c) == norm("Dernier E")), None)
    itv = next((c for c in fdf.columns if norm(c) == norm("Intervalle en jours")), None)
    if de and len(fdf) > 0:
        livres = int((fdf[de].str.strip() == "Envoi Livré").sum())
        kpis["livres"] = livres
        kpis["taux"]   = round(livres / len(fdf) * 100, 1)
    if itv:
        try:
            vals = pd.to_numeric(fdf[itv], errors="coerce").dropna()
            kpis["avg_intervalle"] = round(float(vals.mean()), 1) if len(vals) > 0 else None
        except Exception:
            pass
    if "MAILITM_FID" in fdf.columns:
        kpis["total_ids"] = int(fdf["MAILITM_FID"].nunique())
    else:
        kpis["total_ids"] = kpis["total"]
    ca_col = next((c for c in fdf.columns if norm(c) == "ca"), None)
    if ca_col:
        try:
            kpis["total_ca"] = round(
                float(pd.to_numeric(fdf[ca_col], errors="coerce").sum()), 2)
        except Exception:
            pass
    return kpis

def get_cat_cols(fdf, mx=60):
    return {c: set(fdf[c].str.strip()) for c in fdf.columns if 0 < fdf[c].nunique() <= mx}

def cell_matches(row, col_h, cat_cols):
    for xc, vals in cat_cols.items():
        if col_h in vals and str(row[xc]).strip() == col_h:
            return True
    cc = next((c for c in cat_cols if norm(c) == "crbt"), None)
    if norm(col_h) == "crbt" and cc:
        try: return float(str(row[cc]).replace(",","") or "0") > 0
        except: pass
    if norm(col_h) == "ordinaire" and cc:
        try: return float(str(row[cc]).replace(",","") or "0") == 0
        except: pass
    return False

def find_col_for_values(df, selected_values, max_unique=50):
    real_vals = [v for v in selected_values if v.lower().strip() not in _BLANK_TOKENS]
    if not real_vals:
        return None
    candidates = []
    for col in df.columns:
        unique = set(df[col].str.strip())
        if len(unique) <= max_unique and all(v in unique for v in real_vals):
            candidates.append((len(unique), col))
    if candidates:
        candidates.sort()
        return candidates[0][1]
    return None

def fix_slicer_titles(slicers, df, token=None, dataset_id=None, tbl_name=None):
    BAD = {"clear selections", "clear selection", ""}
    for s in slicers:
        raw = s["title"].lower().strip()
        if raw not in BAD and "clear" not in raw:
            if has_blank_selection(s.get("selected", [])):
                s["dax_filterable"] = True
            elif token and dataset_id and tbl_name:
                s["dax_filterable"] = is_dax_filterable(token, dataset_id, tbl_name, s["title"])
            else:
                s["dax_filterable"] = True
            continue
        # 1. Manual map
        for val in s["selected"]:
            if val in MANUAL_SLICER_MAP:
                s["title"]          = MANUAL_SLICER_MAP[val]
                s["dax_filterable"] = True
                print(f"  ↳ Slicer resolved (manual map): '{s['title']}' ← {s['selected']}")
                break
        if s["title"] and s["title"].lower() not in BAD:
            continue
        # 2. Computed DAX columns
        sel_set = frozenset(s["selected"])
        for val_set, (col_name, _) in _COMPUTED_COLS.items():
            if sel_set <= val_set:
                s["title"]          = col_name
                s["dax_filterable"] = False
                print(f"  ↳ Slicer resolved (computed DAX col): '{col_name}' ← {s['selected']}")
                break
        if s["title"] and s["title"].lower() not in BAD:
            continue
        # 3. DAX auto-discovery
        if token and dataset_id and tbl_name:
            col, filterable = auto_resolve_slicer_dax(token, dataset_id, tbl_name, s["selected"])
            if col:
                filterable = filterable and not has_blank_selection(s["selected"])
                s["title"]          = col
                s["dax_filterable"] = filterable
                mode = "DAX+xlsx" if filterable else "xlsx only"
                print(f"  ↳ Slicer resolved (DAX auto, {mode}): '{col}' ← {s['selected']}")
                continue
        # 4. xlsx auto-detect
        col = find_col_for_values(df, s["selected"])
        if col:
            s["title"]          = col
            s["dax_filterable"] = not has_blank_selection(s["selected"])
            print(f"  ↳ Slicer resolved (xlsx auto): '{col}' ← {s['selected']}")
            continue
        s["title"] = ""
        print(f"  ⚠ Slicer {s['selected']} unresolved — filter skipped")
    return slicers

def extract_measure_base(h):
    for p in ['sum of ', 'average of ', 'avg of ', 'max of ', 'min of ']:
        if h.lower().startswith(p):
            return h[len(p):]
    return None

def get_detail_cols(headers, df):
    result = []; seen = set()
    for h in headers:
        if 'COUNT' in h.upper() and 'MAILITM' in h.upper(): continue
        base = extract_measure_base(h)
        if base is None: base = h
        xlsx_col = next((c for c in df.columns if norm(c) == norm(base)), None)
        if xlsx_col and norm(base) not in seen:
            result.append((base, xlsx_col)); seen.add(norm(base))
    for extra in _NATIONAL_EXTRA_COLS:
        if norm(extra) not in seen:
            xlsx_col = next((c for c in df.columns if norm(c) == norm(extra)), None)
            if xlsx_col:
                result.append((extra, xlsx_col)); seen.add(norm(extra))
    return result

# ── Drill-through ─────────────────────────────────────────────────────────────

def build_national_drill_dax(token, dataset_id, tbl_name, table_meta, slicer_filters, fdf):
    """Exact 2D drill for national tables via DAX (includes column dimension)."""
    filters_str = (",\n    " + ",\n    ".join(slicer_filters)) if slicer_filters else ""
    de_col_xlsx = next((c for c in fdf.columns if norm(c) == norm("Dernier E")), None)
    de_vals     = set(fdf[de_col_xlsx].str.strip()) if de_col_xlsx else set()
    mappings    = {}

    for meta in table_meta:
        if not has_mailitm_count(meta["headers"]): continue
        dim_idx = find_dim_col(meta["headers"])
        if dim_idx is None: continue
        dim_col = meta["headers"][dim_idx]
        non_dim = [h for i, h in enumerate(meta["headers"])
                   if i != dim_idx and "total" not in h.lower()]
        is_categorical = bool(de_vals and any(h in de_vals for h in non_dim))
        is_crbt        = any(norm(h) == "crbt" for h in non_dim)

        if is_categorical:
            # Tableau Region dernier E x Dernier E — query includes status column
            dax = (
                f"EVALUATE CALCULATETABLE(\n"
                f"    SELECTCOLUMNS(\n"
                f"        '{tbl_name}',\n"
                f"        \"DimVal\", '{tbl_name}'[{dim_col}],\n"
                f"        \"ColVal\", '{tbl_name}'[Dernier E],\n"
                f"        \"poids\",  '{tbl_name}'[poids],\n"
                f"        \"CA\",     '{tbl_name}'[CA],\n"
                f"        \"ID\",     '{tbl_name}'[MAILITM_FID]\n"
                f"    ){filters_str}\n"
                f")"
            )
        elif is_crbt:
            # Tableau Bureau depot x CRBT/Ordinaire — split by CRBT>0
            dax = (
                f"EVALUATE CALCULATETABLE(\n"
                f"    SELECTCOLUMNS(\n"
                f"        '{tbl_name}',\n"
                f"        \"DimVal\", '{tbl_name}'[{dim_col}],\n"
                f"        \"CRBT\",   '{tbl_name}'[CRBT],\n"
                f"        \"poids\",  '{tbl_name}'[poids],\n"
                f"        \"CA\",     '{tbl_name}'[CA],\n"
                f"        \"ID\",     '{tbl_name}'[MAILITM_FID]\n"
                f"    ){filters_str}\n"
                f")"
            )
        else:
            continue

        rows = run_dax(token, dataset_id, dax)
        if rows is None:
            print(f"  ❌ DAX failed for national '{meta['key']}'")
            return {}

        flat_map, seen_all, seen_col = {}, {}, {}
        for row in rows:
            dv  = str(row.get("[DimVal]") or row.get("DimVal") or "").strip()
            id_ = str(row.get("[ID]")    or row.get("ID")    or "").strip()
            pwd = str(row.get("[poids]") or row.get("poids") or "")
            if not (dv and id_): continue

            ca_v = str(row.get("[CA]") or row.get("CA") or "")
            if is_categorical:
                cv = str(row.get("[ColVal]") or row.get("ColVal") or "").strip()
                entry = {"id": id_, "poids": pwd, "CA": ca_v, "Dernier E": cv}
            else:
                try:
                    crbt_v = float(str(row.get("[CRBT]") or row.get("CRBT") or "0")
                                   .replace(",", ".").replace(" ", "") or "0")
                except Exception:
                    crbt_v = 0
                cv    = "CRBT" if crbt_v > 0 else "Ordinaire"
                entry = {"id": id_, "poids": pwd, "CA": ca_v}

            # __all__
            if id_ not in seen_all.setdefault(dv, set()):
                seen_all[dv].add(id_)
                flat_map.setdefault(dv, {"__all__": []})["__all__"].append(entry)
            # per-column bucket
            if cv and id_ not in seen_col.setdefault((dv, cv), set()):
                seen_col[(dv, cv)].add(id_)
                flat_map[dv].setdefault(cv, []).append(entry)

        total = sum(len(v["__all__"]) for v in flat_map.values())
        cols  = ["poids", "CA", "Dernier E"] if is_categorical else ["poids", "CA"]
        print(f"  ↳ DAX national '{meta['key']}': {len(flat_map)} dims, {total} IDs")
        mappings[meta["key"]] = {
            "cols": cols, "dim_idx": dim_idx,
            "headers": meta["headers"], "data": flat_map
        }
    return mappings


def build_drill_mappings(token, dataset_id, tables, table_meta, slicers, df):
    tbl_name = find_pbi_table(token, dataset_id) if token and dataset_id else None
    slicers  = fix_slicer_titles(slicers, df, token, dataset_id, tbl_name)

    # ── Apply slicer filters to xlsx ──────────────────────────────────────────
    fdf = df.copy()
    for s in slicers:
        if not s["title"] or not s["selected"]: continue
        col = next((c for c in fdf.columns if norm(c) == norm(s["title"])), None)
        if col:
            before = len(fdf)
            fdf = xlsx_filter(fdf, col, s["selected"])
            print(f"  ↳ xlsx filter: [{col}] = {s['selected']} "
                  f"→ {len(fdf)} rows (was {before})")
        else:
            print(f"  ⚠ [{s['title']}] not in xlsx — skipped")

    if "MAILITM_FID" in fdf.columns:
        valid_ids = set(fdf["MAILITM_FID"].str.strip())
    else:
        valid_ids = None

    # ── Build DAX slicer filters list ─────────────────────────────────────────
    slicer_filters = []
    if tbl_name:
        for s in slicers:
            if not s["title"] or not s["selected"]: continue
            if not s.get("dax_filterable", True): continue
            real_vals = [v for v in s["selected"]
                         if v.lower().strip() not in _BLANK_TOKENS]
            if not real_vals: continue
            vals, col = real_vals, s["title"]
            if len(vals) == 1:
                slicer_filters.append(
                    f"'{tbl_name}'[{col}] = \"{vals[0]}\"")
            else:
                slicer_filters.append(
                    f"'{tbl_name}'[{col}] IN "
                    f"{{{', '.join(chr(34)+v+chr(34) for v in vals)}}}")
            print(f"  ↳ DAX filter: [{col}] = {vals}")

    # ── Detect national context ───────────────────────────────────────────────
    national_ctx = any(
        any(norm(h) == norm(d)
            for h in meta["headers"] for d in _NATIONAL_DIMS)
        for meta in table_meta
    )

    # ── National: dedicated DAX with column dimension (exact IDs) ────────────
    if tbl_name and national_ctx:
        nat = build_national_drill_dax(
            token, dataset_id, tbl_name, table_meta,
            slicer_filters, fdf)
        if nat:
            return nat

    # ── Non-national: existing row-level DAX ─────────────────────────────────
    if tbl_name and not national_ctx:
        mappings, all_ok = {}, True
        for meta, t in zip(table_meta, tables):
            if not has_mailitm_count(meta["headers"]): continue
            dim_idx     = find_dim_col(meta["headers"])
            if dim_idx is None: continue
            dim_col     = meta["headers"][dim_idx]
            detail_cols = get_detail_cols(meta["headers"], fdf)
            print(f"\n  ↳ DAX query '{meta['key']}' "
                  f"(dim: [{dim_col}], detail: {[b for b,_ in detail_cols]})...")
            filters_str = ((",\n    " + ",\n    ".join(slicer_filters))
                           if slicer_filters else "")
            dax = (
                f"EVALUATE\nCALCULATETABLE(\n"
                f"    SELECTCOLUMNS(\n"
                f"        '{tbl_name}',\n"
                f"        \"DimVal\", '{tbl_name}'[{dim_col}],\n"
                f"        \"ID\",     '{tbl_name}'[MAILITM_FID]\n"
                f"    ){filters_str}\n)"
            )
            rows = run_dax(token, dataset_id, dax)
            if rows is None:
                all_ok = False; break

            lookup = {}
            for _, xrow in fdf.iterrows():
                xid = str(xrow["MAILITM_FID"]).strip()
                if xid and xid not in lookup:
                    entry = {b: str(xrow[xc]).strip()
                             for b, xc in detail_cols}
                    lookup[xid] = entry

            flat_map, skipped, seen_per_dv = {}, 0, {}
            for row in rows:
                dv  = row.get("[DimVal]") or row.get("DimVal") or ""
                id_ = row.get("[ID]")    or row.get("ID")    or ""
                if not (dv and id_): continue
                if valid_ids is not None and id_ not in valid_ids:
                    skipped += 1; continue
                if id_ in seen_per_dv.setdefault(dv, set()): continue
                seen_per_dv[dv].add(id_)
                flat_map.setdefault(dv, []).append(
                    {"id": id_,
                     **lookup.get(id_, {b: "" for b, _ in detail_cols})})

            total = sum(len(v) for v in flat_map.values())
            note  = f" ({skipped} post-filtered)" if skipped else ""
            print(f"    ✓ {len(flat_map)} dim vals, {total} unique IDs{note}")
            extra_cols = []
            mappings[meta["key"]] = {
                "cols": [b for b, _ in detail_cols] + extra_cols,
                "data": flat_map}

        if all_ok and mappings:
            return mappings

    # ── xlsx fallback (national 2D cell_matches) ──────────────────────────────
    print("\n  ↳ Using xlsx fallback (2D cell_matches)...")
    mappings = {}
    cat_cols = get_cat_cols(fdf)
    for meta, t in zip(table_meta, tables):
        if not has_mailitm_count(meta["headers"]): continue
        dim_idx = find_dim_col(meta["headers"])
        if dim_idx is None: continue
        disp = meta["headers"][dim_idx]
        col  = next((c for c in fdf.columns if norm(c) == norm(disp)), None)
        if not col:
            print(f"  Warning: '{disp}' not found in xlsx"); continue
        detail_cols  = get_detail_cols(meta["headers"], fdf)
        non_dim_hdrs = [h for i, h in enumerate(meta["headers"])
                        if i != dim_idx and "total" not in h.lower()]
        flat_map, seen_all, seen_cell = {}, {}, {}
        for _, row in fdf.iterrows():
            dv  = str(row[col]).strip()
            id_ = str(row["MAILITM_FID"]).strip()
            if not (dv and id_): continue
            entry = {"id": id_,
                     **{b: str(row[xc]).strip() for b, xc in detail_cols}}
            if id_ not in seen_all.setdefault(dv, set()):
                seen_all[dv].add(id_)
                flat_map.setdefault(dv, {"__all__": []})["__all__"].append(entry)
            for col_h in non_dim_hdrs:
                if cell_matches(row, col_h, cat_cols):
                    if id_ not in seen_cell.setdefault((dv, col_h), set()):
                        seen_cell[(dv, col_h)].add(id_)
                        flat_map[dv].setdefault(col_h, []).append(entry)
        total = sum(len(v["__all__"]) for v in flat_map.values())
        cols_list = [b for b, _ in detail_cols]
        print(f"  ↳ '{meta['key']}': {len(flat_map)} dims, {total} IDs")
        mappings[meta["key"]] = {
            "cols": cols_list, "headers": meta["headers"],
            "dim_idx": dim_idx, "data": flat_map}
    return mappings


def put_totals_last(rows):
    return [r for r in rows if all(c.strip() for c in r)] + \
           [r for r in rows if not all(c.strip() for c in r)]

def find_dim_col(headers):
    for i, h in enumerate(headers):
        if not any(kw in h.upper() for kw in ['SUM','COUNT','AVG','AVERAGE','MAX','MIN']):
            return i
    return None

def has_mailitm_count(headers):
    if any('COUNT' in h.upper() and 'MAILITM' in h.upper() for h in headers):
        return True
    dim_idx = find_dim_col(headers)
    return (dim_idx is not None and
            any(norm(headers[dim_idx]) == norm(d) for d in _NATIONAL_DIMS))

def is_mailitm_table(headers):
    return len(headers) == 1 and 'MAILITM' in headers[0].upper()

JS_CLASSIFY = """() => {
    const c = Array.from(document.querySelectorAll('[data-testid="visual-container"]'));
    return c.map((v, i) => {
        const isTable = !!v.querySelector('[role="columnheader"]');
        const t = v.querySelector('[data-testid="unselectable sidePaneTitle"]')
               || v.querySelector('[class*="visualTitle"] span')
               || v.querySelector('[class*="title"] span');
        return { index: i, type: isTable ? 'table' : 'other',
                 title: t ? t.textContent.trim() : '' };
    });
}"""
JS_SLICERS_GLOBAL = r"""() => {
    const groups = new Map();
    let cbs = Array.from(document.querySelectorAll('[data-testid="slicerCheckbox selected"]'));
    if (!cbs.length)
        cbs = Array.from(document.querySelectorAll('[role="checkbox"][aria-checked="true"]'));
    for (const cb of cbs) {
        const cont = cb.closest('[class*="visualContainer"]')
                  || cb.closest('[class*="visual-container"]')
                  || cb.closest('[data-testid="visual-container"]');
        if (!cont) continue;
        if (!groups.has(cont)) {
            const titleEl =
                cont.querySelector('[data-testid="unselectable sidePaneTitle"]')
             || cont.querySelector('[class*="visualTitle"] span')
             || cont.querySelector('[class*="title"] span')
             || cont.querySelector('[class*="slicerHeader"] .slicer-header-text')
             || cont.querySelector('[class*="slicerHeader"]')
             || cont.querySelector('[aria-label]');
            let title = titleEl
                ? (titleEl.getAttribute('aria-label') || titleEl.textContent || '').trim()
                : '';
            title = title.replace(/clear\s+selections?/gi, '').trim();
            groups.set(cont, { title, selected: [] });
        }
        let text = cb.getAttribute('aria-label') || '';
        if (!text) {
            const row = cb.parentElement;
            const te  = row && row.querySelector('[data-testid="slicerText"]');
            text = te ? te.textContent.trim()
                      : (row ? row.textContent.trim() : cb.textContent.trim());
        }
        if (text === '') text = '(Blank)';
        groups.get(cont).selected.push(text);
    }
    return Array.from(groups.values())
        .filter(g => g.selected.length)
        .map(g => ({ title: g.title, selected: [...new Set(g.selected)] }));
}"""
JS_HEADERS      = """(idx) => {
    const c = document.querySelectorAll('[data-testid="visual-container"]')[idx];
    return Array.from(c.querySelectorAll('[role="columnheader"]'))
        .map(el => el.textContent.trim()).filter(t => t && t !== 'Row Selection');
}"""
JS_VISIBLE_ROWS = """([idx, n]) => {
    const c = document.querySelectorAll('[data-testid="visual-container"]')[idx];
    const cells = Array.from(c.querySelectorAll('[role="gridcell"], [role="rowheader"]'))
        .map(el => el.textContent.trim()).filter(t => t !== 'Select Row' && t !== 'Row Selection');
    const rows = [];
    for (let i = 0; i + n <= cells.length; i += n) rows.push(cells.slice(i, i + n));
    return rows;
}"""
JS_SCROLL       = """([idx, d]) => {
    const c = document.querySelectorAll('[data-testid="visual-container"]')[idx];
    let el = c.querySelector('[role="grid"]');
    if (!el || el.scrollHeight <= el.clientHeight + 5)
        el = Array.from(c.querySelectorAll('div')).find(d => {
            const s = window.getComputedStyle(d);
            return (s.overflowY==='auto'||s.overflowY==='scroll') && d.scrollHeight>d.clientHeight+5;
        });
    if (!el) return { done: true };
    el.scrollTop += d;
    return { done: el.scrollTop + el.clientHeight >= el.scrollHeight - 5 };
}"""
JS_RESET_SCROLL = """(idx) => {
    const c = document.querySelectorAll('[data-testid="visual-container"]')[idx];
    let el = c.querySelector('[role="grid"]');
    if (!el || el.scrollHeight <= el.clientHeight + 5)
        el = Array.from(c.querySelectorAll('div')).find(d => {
            const s = window.getComputedStyle(d);
            return (s.overflowY==='auto'||s.overflowY==='scroll') && d.scrollHeight>d.clientHeight+5;
        });
    if (el) el.scrollTop = 0;
}"""

async def extract_all_rows(page, idx, num_cols):
    all_rows, seen = [], set()
    await page.evaluate(JS_RESET_SCROLL, idx)
    await page.wait_for_timeout(400)
    while True:
        rows = await page.evaluate(JS_VISIBLE_ROWS, [idx, num_cols])
        nf = False
        for row in rows:
            k = "|||".join(row)
            if k not in seen:
                seen.add(k); all_rows.append(row); nf = True
        result = await page.evaluate(JS_SCROLL, [idx, 120])
        await page.wait_for_timeout(300)
        if result.get("done"):
            for row in await page.evaluate(JS_VISIBLE_ROWS, [idx, num_cols]):
                k = "|||".join(row)
                if k not in seen:
                    seen.add(k); all_rows.append(row)
            break
        if not nf: break
    return put_totals_last(all_rows)

# ── HTML builders ─────────────────────────────────────────────────────────────

def _table_html_email(t, n):
    label = t["title"] or f"Tableau {n} — {' | '.join(t['headers'][:3])}"
    th = "".join(
        f'<th style="padding:9px 14px;background:{C_NAVY};color:#fff;'
        f'text-align:left;white-space:nowrap;font-size:12px;">{h}</th>'
        for h in t["headers"])
    rows_html = ""
    for i, row in enumerate(t["rows"]):
        dim_val = row[0].strip() if row else ""
        is_tot = not dim_val or dim_val.lower() == "total"
        if is_tot:
            cells = "".join(
                f'<td style="padding:7px 14px;background:#FFF8E1;font-weight:bold;'
                f'border-top:2px solid {C_YELLOW};font-size:12px;">{c}</td>' for c in row)
        else:
            bg = "#fff" if i % 2 == 0 else C_BG
            cells = "".join(
                f'<td style="padding:7px 14px;border-bottom:1px solid #E4EAF5;'
                f'background:{bg};font-size:12px;">{c}</td>' for c in row)
        rows_html += f"<tr>{cells}</tr>"
    return (
        f'<div style="margin-bottom:28px;">'
        f'<div style="background:{C_NAVY};color:#fff;padding:10px 16px;font-size:13px;'
        f'font-weight:bold;border-radius:6px 6px 0 0;">{label}</div>'
        f'<p style="color:#555;font-size:11px;margin:0;padding:6px 16px;'
        f'background:{C_LIGHT};border:1px solid #D5E1F5;border-top:none;">'
        f'{t["num_rows"]} ligne(s) &times; {t["num_cols"]} colonne(s)</p>'
        f'<div style="overflow-x:auto;border:1px solid #D5E1F5;border-top:none;'
        f'border-radius:0 0 6px 6px;">'
        f'<table style="border-collapse:collapse;width:100%;font-family:Arial,sans-serif;">'
        f'<thead><tr>{th}</tr></thead><tbody>{rows_html}</tbody></table></div></div>'
    )

def build_email_html(filename, slicers, tables, timestamp, has_logo=False, logo_b64=None, kpis=None, section_labels=None):
    if slicers:
        fi_rows = "".join(
            f'<tr><td style="padding:3px 0;font-size:13px;color:#333;">'
            f'<span style="display:inline-block;background:{C_NAVY};color:{C_YELLOW};'
            f'padding:2px 10px;border-radius:3px;font-size:11px;font-weight:bold;'
            f'margin-right:8px;">{s["title"] or "Filtre"}</span>'
            f'{", ".join(v if v.lower() != "(blank)" else "(vide)" for v in s["selected"])}'
            f'</td></tr>'
            for s in slicers if s["selected"])
        filters_html = f'<table cellpadding="0" cellspacing="0" style="width:100%;">{fi_rows}</table>'
    else:
        filters_html = (f'<p style="color:#888;font-style:italic;margin:0;font-size:13px;">'
                        f'Aucun filtre actif.</p>')

    tables_html = ""
    for i, t in enumerate(tables):
        if section_labels and i in section_labels:
            lbl = section_labels[i]
            tables_html += (
                f'<div style="margin:28px 0 16px;border-bottom:3px solid {C_NAVY};padding-bottom:8px;">'
                f'<span style="background:{C_NAVY};color:{C_YELLOW};padding:6px 18px;'
                f'border-radius:6px 6px 0 0;font-size:13px;font-weight:700;letter-spacing:1px;'
                f'text-transform:uppercase;">{lbl}</span></div>'
            )
        tables_html += _table_html_email(t, i+1)

    if logo_b64:
        logo_html = (f'<img src="{logo_b64}" alt="La Poste Tunisienne" '
                     f'style="height:52px;vertical-align:middle;margin-right:14px;">')
    elif has_logo:
        logo_html = (f'<img src="cid:logo_img" alt="La Poste Tunisienne" '
                     f'style="height:52px;vertical-align:middle;margin-right:14px;">')
    else:
        logo_html = ""

    date_fr     = datetime.strptime(timestamp, "%Y-%m-%d %H:%M").strftime("%d/%m/%Y à %H:%M")
    logo_margin = "66px" if (logo_b64 or has_logo) else "2px"

    # KPI block for email
    kpi_email = ""
    if kpis:
        taux      = kpis.get("taux", 0)
        livres    = kpis.get("livres", 0)
        total     = kpis.get("total", 0)
        total_ids = kpis.get("total_ids", total)
        avg       = kpis.get("avg_intervalle")
        avg_s     = f"{avg} j" if avg is not None else "—"
        bar_w     = min(int(taux), 100)
        tid_s     = f"{total_ids:,}".replace(",", " ")
        total_ca  = kpis.get("total_ca")
        ca_s      = (f"{total_ca:,.2f}".replace(",", " ") + " DT")\
                    if total_ca is not None else "—"
        kpi_email = (
            f'<table width="100%" cellpadding="0" cellspacing="0" style="margin-bottom:28px;">'
            f'<tr>'
            f'<td width="34%" style="padding-right:6px;vertical-align:top;">'
            f'<div style="background:{C_LIGHT};border:1px solid #D5E1F5;border-radius:10px;padding:16px 18px;">'
            f'<div style="color:#666;font-size:10px;text-transform:uppercase;letter-spacing:.5px;font-weight:600;margin-bottom:5px;">Taux de livraison</div>'
            f'<div style="color:{C_NAVY};font-size:26px;font-weight:800;line-height:1;margin-bottom:3px;">{taux}%</div>'
            f'<div style="color:#888;font-size:10px;margin-bottom:8px;">{livres} livrés / {total} envois</div>'
            f'<div style="background:#D5E1F5;border-radius:4px;height:7px;">'
            f'<div style="background:{C_YELLOW};border-radius:4px;height:7px;width:{bar_w}%;"></div>'
            f'</div></div></td>'
            f'<td width="25%" style="padding:0 3px;vertical-align:top;">'
            f'<div style="background:{C_LIGHT};border:1px solid #D5E1F5;border-radius:10px;padding:16px 18px;">'
            f'<div style="color:#666;font-size:10px;text-transform:uppercase;letter-spacing:.5px;font-weight:600;margin-bottom:5px;">Délai moyen</div>'
            f'<div style="color:{C_NAVY};font-size:26px;font-weight:800;line-height:1;margin-bottom:3px;">{avg_s}</div>'
            f'<div style="color:#888;font-size:10px;">Intervalle dépôt → livraison</div>'
            f'</div></td>'
            f'<td width="25%" style="padding:0 3px;vertical-align:top;">'
            f'<div style="background:{C_NAVY};border-radius:10px;padding:16px 18px;">'
            f'<div style="color:{C_YELLOW};font-size:10px;text-transform:uppercase;letter-spacing:.5px;font-weight:600;margin-bottom:5px;">Total colis</div>'
            f'<div style="color:#fff;font-size:26px;font-weight:800;line-height:1;margin-bottom:3px;">{tid_s}</div>'
            f'<div style="color:rgba(255,255,255,.6);font-size:10px;">identifiants uniques</div>'
            f'</div></td>'
            f'<td width="25%" style="padding-left:6px;vertical-align:top;">'
            f'<div style="background:#1A6B3A;border-radius:10px;padding:16px 18px;">'
            f'<div style="color:#A8E6C0;font-size:10px;text-transform:uppercase;letter-spacing:.5px;font-weight:600;margin-bottom:5px;">CA Total</div>'
            f'<div style="color:#fff;font-size:26px;font-weight:800;line-height:1;margin-bottom:3px;">{ca_s}</div>'
            f'<div style="color:rgba(255,255,255,.6);font-size:10px;">chiffre d''affaires</div>'
            f'</div></td>'
            f'</tr></table>'
        )

    return f"""<!DOCTYPE html>
<html lang="fr">
<head><meta charset="UTF-8"><meta name="viewport" content="width=device-width,initial-scale=1.0"></head>
<body style="margin:0;padding:0;background:#EAEEF6;font-family:'Segoe UI',Arial,sans-serif;">
<table width="100%" cellpadding="0" cellspacing="0" style="background:#EAEEF6;padding:24px 0;">
<tr><td align="center">
<table width="680" cellpadding="0" cellspacing="0"
  style="background:#fff;border-radius:10px;overflow:hidden;
         box-shadow:0 4px 20px rgba(11,42,111,.15);max-width:680px;">
  <tr><td style="background:{C_NAVY};padding:0;">
    <table width="100%" cellpadding="0" cellspacing="0">
      <tr>
        <td style="padding:22px 30px 18px;vertical-align:middle;">
          {logo_html}
          <span style="color:#fff;font-size:20px;font-weight:700;letter-spacing:.5px;vertical-align:middle;">
            LA POSTE TUNISIENNE</span><br>
          <span style="color:{C_YELLOW};font-size:11px;letter-spacing:2px;text-transform:uppercase;
                        margin-left:{logo_margin};">
            Rapport Automatisé &mdash; National</span>
        </td>
        <td style="padding:22px 30px 18px;text-align:right;white-space:nowrap;vertical-align:middle;">
          <span style="background:{C_YELLOW};color:{C_NAVY};padding:6px 14px;
                        border-radius:20px;font-size:12px;font-weight:700;">{date_fr}</span>
        </td>
      </tr>
    </table>
    <div style="height:4px;background:{C_YELLOW};"></div>
  </td></tr>
  <tr><td style="padding:32px 36px 24px;">
    <p style="color:{C_NAVY};font-size:15px;font-weight:600;margin:0 0 16px;">Madame, Monsieur,</p>
    <p style="color:#444;font-size:14px;line-height:1.75;margin:0 0 28px;">
      Veuillez trouver ci-dessous le <strong>rapport automatisé de suivi National</strong>,
      généré le <strong>{date_fr}</strong>. Un fichier HTML interactif est joint.</p>
    {kpi_email}
    <table width="100%" cellpadding="0" cellspacing="0" style="margin-bottom:28px;">
      <tr><td style="background:{C_LIGHT};border-left:4px solid {C_NAVY};
                      padding:14px 18px;border-radius:0 6px 6px 0;">
        <div style="color:{C_NAVY};font-weight:700;font-size:12px;
                     text-transform:uppercase;letter-spacing:.5px;margin-bottom:10px;">
          Filtres appliqués</div>{filters_html}
      </td></tr>
    </table>
    <div style="margin-bottom:32px;">
      <div style="background:{C_NAVY};color:#fff;padding:10px 16px;font-size:13px;
                   font-weight:700;border-radius:6px 6px 0 0;">
        Aperçu du tableau de bord Power BI</div>
      <img src="cid:report_img" style="width:100%;display:block;border:1px solid #D5E1F5;
               border-top:none;border-radius:0 0 6px 6px;">
    </div>
    <div style="color:{C_NAVY};font-size:14px;font-weight:700;
                 border-bottom:2px solid {C_YELLOW};padding-bottom:8px;margin-bottom:22px;">
      Données détaillées</div>
    {tables_html}
    <p style="color:#888;font-size:12px;line-height:1.6;margin:24px 0 0;">
      Cordialement,<br>
      <strong style="color:{C_NAVY};">Direction des Systèmes d'Information</strong><br>
      La Poste Tunisienne</p>
  </td></tr>
  <tr><td style="background:{C_NAVY};padding:0;">
    <div style="height:3px;background:{C_YELLOW};"></div>
    <table width="100%" cellpadding="0" cellspacing="0">
      <tr><td style="padding:18px 30px;text-align:center;">
        <div style="color:{C_YELLOW};font-size:13px;font-weight:700;letter-spacing:1px;margin-bottom:4px;">
          LA POSTE TUNISIENNE</div>
        <div style="color:rgba(255,255,255,.55);font-size:11px;">
          Direction des Systèmes d'Information &bull; Rapport généré automatiquement</div>
      </td></tr>
    </table>
  </td></tr>
</table>
</td></tr>
</table>
</body></html>"""

def build_interactive_html(slicers, tables, drill_mappings, timestamp, logo_b64=None, kpis=None, auto_refresh=False, section_labels=None):
    fi = ""
    if slicers:
        items = "".join(
            f'<li><strong style="color:{C_NAVY};">{s["title"] or "Filtre"}</strong>'
            f' : {", ".join(v if v.lower() != "(blank)" else "(vide)" for v in s["selected"])}</li>'
            for s in slicers if s["selected"])
        fi = (f'<div class="fbox"><strong>Filtres appliqués :</strong>'
              f'<ul style="margin:8px 0 0;padding-left:20px;">{items}</ul></div>')
    kpi_html = ""
    if kpis:
        taux      = kpis.get("taux", 0)
        livres    = kpis.get("livres", 0)
        total     = kpis.get("total", 0)
        total_ids = kpis.get("total_ids", total)
        avg       = kpis.get("avg_intervalle")
        avg_s     = f"{avg} j" if avg is not None else "—"
        bar_w     = min(int(taux), 100)
        tid_s     = f"{total_ids:,}".replace(",", " ")
        total_ca  = kpis.get("total_ca")
        ca_s      = (f"{total_ca:,.2f}".replace(",", " ") + " DT")\
                    if total_ca is not None else "—"
        kpi_html = (
            f'<div class="kpi-row">'
            f'<div class="kpi-card">'
            f'<div class="kpi-icon">&#128230;</div>'
            f'<div class="kpi-body">'
            f'<div class="kpi-label">Taux de livraison</div>'
            f'<div class="kpi-val">{taux}%</div>'
            f'<div class="kpi-sub">{livres} livrés / {total} envois</div>'
            f'<div class="kpi-bar-bg"><div class="kpi-bar-fill" style="width:{bar_w}%"></div></div>'
            f'</div></div>'
            f'<div class="kpi-card">'
            f'<div class="kpi-icon">&#9201;</div>'
            f'<div class="kpi-body">'
            f'<div class="kpi-label">Délai moyen</div>'
            f'<div class="kpi-val">{avg_s}</div>'
            f'<div class="kpi-sub">Intervalle dépôt → livraison</div>'
            f'</div></div>'
            f'<div class="kpi-card" style="background:{C_NAVY};border-color:{C_NAVY};">'
            f'<div class="kpi-icon" style="font-size:24px;">&#128221;</div>'
            f'<div class="kpi-body">'
            f'<div class="kpi-label" style="color:{C_YELLOW};">Total colis</div>'
            f'<div class="kpi-val" style="color:#fff;">{tid_s}</div>'
            f'<div class="kpi-sub" style="color:rgba(255,255,255,.6);">identifiants uniques</div>'
            f'</div></div>'
            f'<div class="kpi-card" style="background:#1A6B3A;border-color:#1A6B3A;">'
            f'<div class="kpi-icon" style="font-size:24px;">&#128176;</div>'
            f'<div class="kpi-body">'
            f'<div class="kpi-label" style="color:#A8E6C0;">CA Total</div>'
            f'<div class="kpi-val" style="color:#fff;">{ca_s}</div>'
            f'<div class="kpi-sub" style="color:rgba(255,255,255,.6);">chiffre d\'affaires</div>'
            f'</div></div>'
            f'</div>'
        )

    th_html = ""
    for n, t in enumerate(tables):
        if section_labels and n in section_labels:
            lbl = section_labels[n]
            th_html += (
                f'<div class="section-hdr">'
                f'<span class="section-hdr-text">{lbl}</span></div>'
            )
        tk         = f"t{n}"
        dm         = drill_mappings.get(tk, {})
        dm_data    = dm.get("data", {}) if isinstance(dm, dict) else {}
        has_drill  = bool(dm_data)
        dim_idx_dm = dm.get("dim_idx", 0) if isinstance(dm, dict) else 0
        label      = t["title"] or f"Tableau {n+1}"
        hint       = ' <span class="hint">— cliquez pour explorer</span>' if has_drill else ""
        pbi_tot_idx = set(
            ri for ri, row in enumerate(t["rows"])
            if not (row[dim_idx_dm].strip() if dim_idx_dm < len(row) else "")
            or (row[dim_idx_dm].strip().lower() if dim_idx_dm < len(row) else "") == "total"
        )
        col_totals = [None] * len(t["headers"])
        for ci in range(len(t["headers"])):
            vals = []
            for ri, row in enumerate(t["rows"]):
                if ri in pbi_tot_idx: continue
                if ci < len(row):
                    try: vals.append(float(row[ci].replace(" ","").replace(" ","").replace(",",".")))
                    except: pass
            if vals: col_totals[ci] = str(int(round(sum(vals))))
        th = "".join(f'<th>{h}</th>' for h in t["headers"])
        # Compute CA per dim value from drill entries
        ca_by_dv = {}
        if has_drill and dm_data:
            for _dv, _dd in dm_data.items():
                _ents = _dd.get("__all__", []) if isinstance(_dd, dict) else _dd
                _s = 0.0
                for _e in _ents:
                    try: _s += float(str(_e.get("CA") or "0").replace(",","").replace(" ","") or "0")
                    except: pass
                if _s: ca_by_dv[_dv] = round(_s, 2)
        show_ca = bool(ca_by_dv)
        if show_ca:
            th += '<th style="color:#1A6B3A;font-weight:700;white-space:nowrap;">CA (DT)</th>'
        rows_html = ""
        for ri, row in enumerate(t["rows"]):
            is_tot = ri in pbi_tot_idx
            cells = ""
            dv_js  = (row[dim_idx_dm].replace("\\", "\\\\").replace("'", "\\'")
                     if dim_idx_dm < len(row) else "")
            for ci, val in enumerate(row):
                hdr = t["headers"][ci] if ci < len(t["headers"]) else ""
                if is_tot:
                    cells += f'<td class="tot-cell">{val}</td>'
                elif has_drill:
                    if ci == dim_idx_dm:
                        cells += f'<td class="cl dim-cell" onclick="drill(\'{tk}\',\'{dv_js}\',undefined)">{val}</td>'
                    elif "total" not in hdr.lower():
                        hdr_js = hdr.replace("\\", "\\\\").replace("'", "\\'")
                        cells += f'<td class="cl dat-cell" onclick="drill(\'{tk}\',\'{dv_js}\',\'{hdr_js}\')">{val}</td>'
                    else:
                        cells += f'<td class="cl" onclick="drill(\'{tk}\',\'{dv_js}\',undefined)">{val}</td>'
                else:
                    cells += f'<td>{val}</td>'
            if show_ca:
                _dv_k = row[dim_idx_dm].strip() if dim_idx_dm < len(row) else ""
                _ca_v = ca_by_dv.get(_dv_k, 0)
                if is_tot:
                    _ca_s2 = f"{sum(ca_by_dv.values()):.2f}" if ca_by_dv else "—"
                    cells += f'<td class="tot-cell">{_ca_s2}</td>'
                elif _ca_v:
                    cells += f'<td class="ca-cell">{_ca_v:.2f}</td>'
                else:
                    cells += '<td class="ca-cell">—</td>'
            rows_html += f'<tr class="{"tot" if is_tot else ""}">{cells}</tr>'

        foot_cells = ""
        for ci, tot in enumerate(col_totals):
            if ci == 0 and not tot:
                foot_cells += '<td class="foot-total"><strong>Total</strong></td>'
            else:
                foot_cells += f'<td class="foot-total">{tot or "—"}</td>'
        if show_ca and ca_by_dv:
            _ca_gt = round(sum(ca_by_dv.values()), 2)
            foot_cells += f'<td class="foot-total" style="color:#28A745;">{_ca_gt:.2f}</td>'
        tfoot = f"<tfoot><tr>{foot_cells}</tr></tfoot>" if any(col_totals) else ""
        th_html += (
            f'<h3>{label}{hint}</h3>'
            f'<p class="meta">{t["num_rows"]} lignes × {t["num_cols"]} colonnes</p>'
            f'<div class="tw"><table id="{tk}"><thead><tr>{th}</tr></thead>'
            f'<tbody>{rows_html}</tbody>{tfoot}</table></div>'
        )
    js_data  = json.dumps(drill_mappings, ensure_ascii=False)
    date_fr  = datetime.strptime(timestamp, "%Y-%m-%d %H:%M").strftime("%d/%m/%Y à %H:%M")
    logo_tag = (f'<img src="{logo_b64}" alt="La Poste Tunisienne" '
                f'style="height:48px;margin-right:14px;vertical-align:middle;">'
                if logo_b64 else "")
    refresh  = '<meta http-equiv="refresh" content="6">' if auto_refresh else ""
    html = f"""<!DOCTYPE html><html lang="fr"><head><meta charset="UTF-8">{refresh}
<title>Rapport National — La Poste Tunisienne</title>
<style>
*{{box-sizing:border-box}}
body{{font-family:'Segoe UI',Arial,sans-serif;margin:0;padding:0;background:{C_BG};color:#333;font-size:14px}}
.wrapper{{max-width:1400px;margin:auto;padding:24px}}
.page-header{{background:{C_NAVY};border-radius:10px 10px 0 0;overflow:hidden}}
.page-header .top{{display:flex;align-items:center;justify-content:space-between;padding:20px 28px 16px}}
.brand{{color:#fff;font-size:22px;font-weight:700}}.sub{{color:{C_YELLOW};font-size:11px;letter-spacing:2px;text-transform:uppercase;margin-top:4px}}
.badge{{background:{C_YELLOW};color:{C_NAVY};padding:6px 16px;border-radius:20px;font-size:12px;font-weight:700;white-space:nowrap}}
.accent-bar{{height:4px;background:{C_YELLOW}}}
.content{{background:#fff;border-radius:0 0 10px 10px;padding:28px;box-shadow:0 4px 20px rgba(11,42,111,.1);margin-bottom:24px}}
h3{{color:{C_NAVY};border-left:4px solid {C_YELLOW};padding-left:12px;margin-top:32px;margin-bottom:6px;font-size:15px}}
.hint{{color:{C_YELLOW};font-size:11px;font-weight:normal;font-style:italic}}
.meta{{color:#666;font-size:12px;margin:2px 0 8px}}.tw{{overflow-x:auto}}
table{{border-collapse:collapse;width:100%;font-size:13px;margin-bottom:10px;background:white;border-radius:6px;box-shadow:0 1px 6px rgba(11,42,111,.08)}}
th{{background:{C_NAVY};color:white;padding:9px 14px;text-align:left;white-space:nowrap;font-size:12px}}
td{{padding:7px 14px;border-bottom:1px solid #E4EAF5;font-size:13px}}
tr:nth-child(even) td{{background:{C_BG}}}
.tot td,.tot-cell{{background:#FFF8E1;font-weight:bold;border-top:2px solid {C_YELLOW}}}.ca-cell{{color:#1A6B3A;font-weight:600;}}
tfoot td,.foot-total{{background:{C_NAVY}!important;color:#fff!important;font-weight:700;font-size:12px;padding:7px 14px;border-top:3px solid {C_YELLOW}}}
.cl{{cursor:pointer}}.dim-cell:hover td,.dim-cell:hover{{background:#FFF0C8!important;font-weight:600}}
.dat-cell:hover{{background:{C_LIGHT}!important}}
.fbox{{background:{C_LIGHT};border-left:4px solid {C_NAVY};padding:14px 18px;margin-bottom:24px;border-radius:0 6px 6px 0}}
.kpi-row{{display:flex;gap:16px;margin-bottom:28px;flex-wrap:wrap}}
.kpi-card{{flex:1;min-width:200px;background:{C_LIGHT};border:1px solid #D5E1F5;border-radius:10px;padding:18px 20px;display:flex;gap:14px;align-items:flex-start}}
.kpi-icon{{font-size:28px;line-height:1}}.kpi-body{{flex:1}}
.kpi-label{{color:#666;font-size:11px;text-transform:uppercase;letter-spacing:.5px;font-weight:600;margin-bottom:4px}}
.kpi-val{{color:{C_NAVY};font-size:26px;font-weight:800;line-height:1;margin-bottom:4px}}
.kpi-sub{{color:#888;font-size:11px}}
.kpi-bar-bg{{background:#D5E1F5;border-radius:4px;height:6px;margin-top:8px}}
.kpi-bar-fill{{background:{C_YELLOW};border-radius:4px;height:6px}}
#panel{{display:none;position:fixed;bottom:0;left:0;right:0;background:white;border-top:4px solid {C_YELLOW};padding:16px 28px;box-shadow:0 -4px 24px rgba(11,42,111,.2);max-height:50vh;overflow-y:auto;z-index:999}}
#panel h3{{margin:0 0 4px;border-left:4px solid {C_YELLOW};padding-left:10px;font-size:15px}}
#xbtn{{float:right;cursor:pointer;font-size:24px;color:#999;background:none;border:none;line-height:1}}
.tag{{display:inline-block;background:{C_NAVY};color:{C_YELLOW};padding:3px 12px;border-radius:12px;font-size:12px;margin-left:6px;font-weight:600}}
.col-tag{{display:inline-block;background:{C_YELLOW};color:{C_NAVY};padding:2px 10px;border-radius:12px;font-size:11px;margin-left:4px;font-weight:600;vertical-align:middle}}
#ptable{{width:auto;min-width:420px;box-shadow:none}}
#ptable th{{background:#1A3D8A;font-size:12px;padding:7px 12px}}
#ptable td{{font-size:12px;padding:6px 12px}}
.no-data{{color:#999;font-style:italic}}
.section-hdr{{margin:32px 0 16px;border-bottom:3px solid {C_NAVY};padding-bottom:0;}}
.section-hdr-text{{background:{C_NAVY};color:{C_YELLOW};padding:7px 20px;display:inline-block;border-radius:6px 6px 0 0;font-size:13px;font-weight:700;letter-spacing:1px;text-transform:uppercase;}}
footer{{text-align:center;padding:16px;background:{C_NAVY};border-radius:10px;color:rgba(255,255,255,.5);font-size:11px;margin-top:4px}}
footer span{{color:{C_YELLOW};font-weight:700}}
</style></head><body>
<div class="wrapper">
  <div class="page-header">
    <div class="top">
      <div style="display:flex;align-items:center;">{logo_tag}
        <div><div class="brand">LA POSTE TUNISIENNE</div>
             <div class="sub">Rapport National &mdash; Interactif</div></div>
      </div>
      <div class="badge">{date_fr}</div>
    </div>
    <div class="accent-bar"></div>
  </div>
  <div class="content">KPIS_BLOCKFILTER_INFOTABLES_HTML</div>
  <footer><span>LA POSTE TUNISIENNE</span> &bull; Direction des Systèmes d'Information</footer>
</div>
<div id="panel">
  <button id="xbtn" onclick="closeP()">&#x2715;</button>
  <h3>Identifiants &mdash; <span id="lbl"></span></h3>
  <p id="cnt" style="color:#666;font-size:12px;margin:4px 0 10px"></p>
  <div style="overflow-x:auto">
    <table id="ptable"><thead id="phead"></thead><tbody id="pbody"></tbody></table>
  </div>
</div>
<script>
const D=DRILL_DATA;
function drill(tk,dv,colH){{
  const tm=D[tk]||{{}};const cols=tm.cols||[];
  const ddata=(tm.data||{{}})[dv]||{{}};
  const key=colH||'__all__';
  let rows=ddata[key]||(Array.isArray(ddata)?ddata:[]);
  rows=Array.isArray(rows)?rows:[];
  let label='<span class="tag">'+dv+'</span>';
  if(colH)label+='<span class="col-tag">'+colH+'</span>';
  document.getElementById('lbl').innerHTML=label;
  document.getElementById('cnt').textContent=rows.length+' identifiant(s)';
  document.getElementById('phead').innerHTML='<tr><th>MAILITM_FID</th>'+cols.map(c=>'<th>'+c+'</th>').join('')+'</tr>';
  if(rows.length){{
    document.getElementById('pbody').innerHTML=rows.map(r=>{{
      const id=typeof r==='string'?r:(r.id||'');
      const extras=cols.map(c=>{{

        return '<td>'+(r[c]!==undefined?r[c]:'')+'</td>';
      }}).join('');
      return '<tr><td>'+id+'</td>'+extras+'</tr>';
    }}).join('');
  }}else{{
    document.getElementById('pbody').innerHTML='<tr><td class="no-data" colspan="'+(cols.length+1)+'">Aucun identifiant.</td></tr>';
  }}
  document.getElementById('panel').style.display='block';
  document.getElementById('panel').scrollIntoView({{behavior:'smooth',block:'end'}});
}}
function closeP(){{document.getElementById('panel').style.display='none';}}
</script></body></html>"""
    return (html.replace("KPIS_BLOCK", kpi_html)
                .replace("FILTER_INFO", fi)
                .replace("TABLES_HTML", th_html)
                .replace("DRILL_DATA", js_data))

async def run_all():
    token_holder = []
    _schema_cache.clear()

    async with async_playwright() as p:
        context = await p.chromium.launch_persistent_context(
            user_data_dir=PROFILE_DIR, headless=True,
            viewport={"width": 1920, "height": 1080}, device_scale_factor=2,
        )
        page = await context.new_page()

        def on_request(request):
            auth = request.headers.get("authorization", "")
            if (auth.startswith("Bearer ") and not token_holder and
                    any(d in request.url for d in ["powerbi.com", "analysis.windows.net"])):
                token_holder.append(auth[7:])

        page.on("request", on_request)
        await page.goto(REPORT_URL, timeout=60000)
        try:
            await page.wait_for_load_state("load", timeout=15000)
        except Exception: pass

        if "app.powerbi.com" not in page.url:
            print("❌ Session expired — re-run Step 1.")
            await context.close(); return

        await page.wait_for_timeout(8000)

        bearer_token = token_holder[0] if token_holder else None
        print(f"  {'✅ Bearer token captured' if bearer_token else '❌ Bearer token NOT found'}")

        slicers = await page.evaluate(JS_SLICERS_GLOBAL)
        print(f"  ↳ {len(slicers)} slicer(s): {[(s['title'], s['selected']) for s in slicers]}")

        visuals = await page.evaluate(JS_CLASSIFY)
        tables, table_meta = [], []
        for v in visuals:
            if v["type"] != "table": continue
            headers = await page.evaluate(JS_HEADERS, v["index"])
            if not headers or is_mailitm_table(headers): continue
            label = v["title"] or f"Tableau — {' | '.join(headers[:3])}"
            print(f"  ↳ Extracting '{label}'...")
            rows = await extract_all_rows(page, v["index"], len(headers))
            tkey = f"t{len(tables)}"
            tables.append({"title": v["title"], "headers": headers,
                            "rows": rows, "num_rows": len(rows), "num_cols": len(headers)})
            table_meta.append({"key": tkey, "idx": v["index"], "headers": headers})
            print(f"    ✓ {len(rows)} rows")

        filename = f"report_{datetime.now().strftime('%Y%m%d_%H%M%S')}.png"
        await page.screenshot(path=filename, full_page=False)
        await context.close()

    print("\n── Building drill-through mappings ─────────────")
    try:
        df = load_xlsx_df()
    except Exception as e:
        print(f"  ⚠ Could not load xlsx: {e}")
        df = pd.DataFrame()

    dataset_id = get_dataset_id(bearer_token) if bearer_token else None
    drill_mappings = build_drill_mappings(
        bearer_token, dataset_id, tables, table_meta, slicers, df)

    # ── KPIs (computed from filtered xlsx) ──
    fdf2 = df.copy()
    for s in slicers:
        if not s.get("title") or not s.get("selected"): continue
        col2 = next((c for c in fdf2.columns if norm(c) == norm(s["title"])), None)
        if col2:
            fdf2 = xlsx_filter(fdf2, col2, s["selected"])
    kpis = compute_kpis(fdf2)

    for k, v in drill_mappings.items():
        cols = v.get("cols", []) if isinstance(v, dict) else []
        data = v.get("data", {}) if isinstance(v, dict) else {}
        print(f"  [{k}] {len(data)} dim vals | panel cols: {cols}")

    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M")
    has_logo  = os.path.exists(LOGO_PATH)
    logo_b64  = get_logo_b64()
    print(f"\n✅ Screenshot : {filename}")
    print(f"  Logo : {'✅ base64 (email + HTML)' if logo_b64 else '⚠ logo_laposte.jpg introuvable'}")

    html_filename = filename.replace(".png", "_interactif.html")
    with open(html_filename, "w", encoding="utf-8") as f:
        f.write(build_interactive_html(slicers, tables, drill_mappings, timestamp, logo_b64, kpis))
    print(f"✅ HTML interactif : {html_filename}")

    # Logo embedded as base64 in the email body — no CID attachment needed
    email_html = build_email_html(filename, slicers, tables, timestamp, has_logo, logo_b64, kpis)
    msg = MIMEMultipart("related")
    msg["From"]    = EMAIL_FROM
    msg["To"]      = ", ".join(EMAIL_TO)
    msg["Subject"] = (f"[La Poste Tunisienne] Rapport National — "
                      f"{datetime.strptime(timestamp, '%Y-%m-%d %H:%M').strftime('%d/%m/%Y')}")
    msg.attach(MIMEText(email_html, "html", "utf-8"))

    with open(filename, "rb") as f:
        img = MIMEImage(f.read())
    img.add_header("Content-ID", "<report_img>")
    img.add_header("Content-Disposition", "inline", filename=filename)
    msg.attach(img)

    with open(html_filename, "rb") as f:
        att = MIMEBase("text", "html")
        att.set_payload(f.read())
    encoders.encode_base64(att)
    att.add_header("Content-Disposition", "attachment",
                   filename=os.path.basename(html_filename))
    msg.attach(att)

    with smtplib.SMTP_SSL("smtp.gmail.com", 465) as server:
        server.login(EMAIL_FROM, EMAIL_PASSWORD)
        server.sendmail(EMAIL_FROM, EMAIL_TO, msg.as_string())
    print(f"📧 Email envoyé → {EMAIL_TO}")

def run():
    loop = asyncio.ProactorEventLoop() if sys.platform == "win32" else asyncio.new_event_loop()
    asyncio.set_event_loop(loop)
    try:
        loop.run_until_complete(run_all())
    finally:
        loop.close()

thread = threading.Thread(target=run)
thread.start()
thread.join()

Visuels détectés : 0

Headers des tables :

Premières lignes de chaque table :

Slicers détectés :
  (aucun slicer coché)
  ✅ Bearer token captured
  ↳ 0 slicer(s): []

── Building drill-through mappings ─────────────
  ↳ Categorie_Bureau_Dernier_E_nle computed from [Bureau dernier E]
  ↳ Dataset ID: 96786199-742d-4006-b6c5-acf1ce393a8a
  ↳ Categorie_Bureau_Dernier_E_nle computed from [Bureau dernier E]
Watch mode actif — ouvrez national_live.html dans votre navigateur
Modifiez les slicers dans Power BI pour régénérer.
Arrêtez avec Stop (kernel Jupyter)


  ↳ Using xlsx fallback (2D cell_matches)...
  ✅ 2026-06-14 18:25 — HTML mis à jour | livraison: 61.1% | délai moy: 2.8j

  ↳ Using xlsx fallback (2D cell_matches)...

✅ Screenshot : report_20260614_182535.png
  Logo : ✅ base64 (email + HTML)
✅ HTML interactif : report_20260614_182535_interactif.html
📧 Email envoyé → ['ala-eddine.bouzid@esprit.tn']


In [6]:
# ── STEP 3: Email automatique par région — DÉPÔT + LIVRAISON ──────────────────
# Prérequis : avoir exécuté Step 2 au moins une fois (fonctions chargées en mémoire).
# Le slicer "Catégorie bureau = Agences" DOIT être déjà coché dans Power BI.
# Ce script change uniquement le slicer "Region Depot" pour chaque région.
#
# Structure de l'email par région :
#   Section DÉPÔT   → National Tableau 1  (Bureau depot × CRBT/Ord/Total/CA)
#                   + Export    Tableau    (même structure, données export.xlsx)
#   Section LIVRAISON → National Tableau 2  (Region dernier E × Events/Total/CA)
#                     + Import    Tableau    (Bureau dernier E × Events/Total)

# ╔══════════════════════════════════════════════════════╗
# ║  EMAILS PAR RÉGION — REMPLISSEZ CE DICTIONNAIRE     ║
# ║  Laissez "" pour utiliser l'adresse par défaut      ║
# ╚══════════════════════════════════════════════════════╝
REGION_EMAILS = {
    "ARIANA":      "",
    "BEJA":        "",
    "BEN AROUS":   "",
    "BIZERTE":     "",
    "GABES":       "",
    "GAFSA":       "",
    "JENDOUBA":    "",
    "KAIROUAN":    "",
    "KASSERINE":   "",
    "KEBILI":      "",
    "KEF":         "",
    "MAHDIA":      "",
    "MANOUBA":     "",
    "MEDENINE":    "",
    "MONASTIR":    "",
    "NABEUL":      "",
    "SFAX":        "",
    "SIDI BOUZID": "",
    "SILIANA":     "",
    "SOUSSE":      "",
    "TATAOUINE":   "",
    "TOZEUR":      "",
    "TUNIS":       "",
    "ZAGHOUAN":    "",
}
DEFAULT_EMAIL  = "ala-eddine.bouzid@esprit.tn"
FIXED_CATEGORY = "Agences"
IMPORT_XLS_PATH = r"c:\Users\USER\OneDrive - ESPRIT\Bureau\Code_PFE_Master\import.xls"
# ─────────────────────────────────────────────────────────────────────────────

_REGION_NAMES_UPPER = [r.upper() for r in _REGIONS]

# Section labels for HTML rendering
SECTION_LABELS = {0: "DEPOT", 2: "LIVRAISON"}

# ── JS pour interagir avec les slicers ────────────────────────────────────────

JS_FIND_REGION_SLICER = """(regionNames) => {
    const root = document.body;

    function matchesRegionSlicer(el) {
        const texts1 = Array.from(el.querySelectorAll('[data-testid="slicerText"]'))
            .map(e => e.textContent.trim().toUpperCase());
        if (texts1.some(t => regionNames.includes(t))) return true;

        const leaves = Array.from(el.querySelectorAll('*')).filter(e =>
            e.children.length === 0 && regionNames.includes(e.textContent.trim().toUpperCase()));
        if (leaves.length > 0) return true;

        const ariaOk = Array.from(el.querySelectorAll('[aria-label]')).some(e =>
            regionNames.includes((e.getAttribute('aria-label') || '').trim().toUpperCase()));
        if (ariaOk) return true;

        const checked = Array.from(el.querySelectorAll('[aria-checked]'));
        for (const cb of checked) {
            const label = cb.getAttribute('aria-label') || cb.textContent || '';
            if (regionNames.includes(label.trim().toUpperCase())) return true;
        }
        return false;
    }

    const containers = Array.from(root.querySelectorAll('[data-testid="visual-container"]'));
    if (containers.length > 0) {
        for (let i = 0; i < containers.length; i++) {
            if (matchesRegionSlicer(containers[i])) return i;
        }
        return -1;
    }

    for (const sel of [
        '[data-element-type="visual"]',
        '.visual-container',
        '[class*="visual-container"]',
        '[class*="visualContainer"]',
    ]) {
        const cs = Array.from(root.querySelectorAll(sel));
        if (cs.length > 0) {
            for (let i = 0; i < cs.length; i++) {
                if (matchesRegionSlicer(cs[i])) return i;
            }
        }
    }
    return -1;
}"""

JS_CLEAR_SLICER = """(idx) => {
    const c = Array.from(document.querySelectorAll('[data-testid="visual-container"]'))[idx];
    if (!c) return false;
    const btn = c.querySelector('[aria-label="Clear selections"]')
             || c.querySelector('[aria-label="Effacer les sélections"]')
             || c.querySelector('[title="Clear selections"]')
             || Array.from(c.querySelectorAll('button,[role="button"]')).find(b =>
                    (b.getAttribute('aria-label')||b.title||'').toLowerCase().includes('clear') ||
                    (b.getAttribute('aria-label')||b.title||'').toLowerCase().includes('effacer'));
    if (btn) { btn.click(); return true; }
    return false;
}"""

JS_SLICER_SCROLL_TOP = """(idx) => {
    const c = Array.from(document.querySelectorAll('[data-testid="visual-container"]'))[idx];
    if (!c) return;
    const el = Array.from(c.querySelectorAll('div')).find(d => {
        const s = window.getComputedStyle(d);
        return (s.overflowY==='auto'||s.overflowY==='scroll') && d.scrollHeight>d.clientHeight+5;
    });
    if (el) el.scrollTop = 0;
}"""

JS_TRY_SELECT_IN_SLICER = """([idx, value]) => {
    const c = Array.from(document.querySelectorAll('[data-testid="visual-container"]'))[idx];
    if (!c) return false;
    const upper = value.toUpperCase();
    for (const item of c.querySelectorAll('[data-testid="slicerText"]')) {
        if (item.textContent.trim().toUpperCase() === upper) {
            const row = item.closest('[role="option"]')
                     || item.closest('[class*="row"]')
                     || item.parentElement;
            if (row) { row.click(); return true; }
            item.click(); return true;
        }
    }
    for (const el of c.querySelectorAll('*')) {
        if (el.children.length === 0 && el.textContent.trim().toUpperCase() === upper) {
            const row = el.closest('[role="option"]') || el.closest('[role="listitem"]')
                     || el.closest('[class*="row"]') || el.parentElement;
            if (row) { row.click(); return true; }
            el.click(); return true;
        }
    }
    for (const el of c.querySelectorAll('[aria-label]')) {
        if ((el.getAttribute('aria-label')||'').trim().toUpperCase() === upper) {
            el.click(); return true;
        }
    }
    return false;
}"""

JS_SCROLL_SLICER = """([idx, amount]) => {
    const c = Array.from(document.querySelectorAll('[data-testid="visual-container"]'))[idx];
    if (!c) return { done: true };
    const el = Array.from(c.querySelectorAll('div')).find(d => {
        const s = window.getComputedStyle(d);
        return (s.overflowY==='auto'||s.overflowY==='scroll') && d.scrollHeight>d.clientHeight+5;
    });
    if (!el) return { done: true };
    el.scrollTop += amount;
    return { done: el.scrollTop + el.clientHeight >= el.scrollHeight - 5 };
}"""

JS_DUMP_CONTAINERS = """() => {
    const cs = Array.from(document.querySelectorAll('[data-testid="visual-container"]'));
    return cs.map((c, i) => ({
        idx: i,
        slicerTexts: Array.from(c.querySelectorAll('[data-testid="slicerText"]'))
                         .map(e => e.textContent.trim()).slice(0, 5),
        leafTexts: Array.from(c.querySelectorAll('*'))
                       .filter(e => e.children.length === 0 && e.textContent.trim().length > 0
                                 && e.textContent.trim().length < 40)
                       .map(e => e.textContent.trim()).slice(0, 5),
        ariaLabels: Array.from(c.querySelectorAll('[aria-label]'))
                        .map(e => e.getAttribute('aria-label')).slice(0, 3),
    }));
}"""

# ── Helpers: table type identification ───────────────────────────────────────

def is_depot_table(headers):
    return any(norm(h) == norm("Bureau depot") for h in headers)

def is_livraison_table(headers):
    return any(norm(h) == norm("Region dernier E") for h in headers)

# ── DOM slicer helpers ────────────────────────────────────────────────────────

async def set_region_slicer(page, slicer_idx, region):
    cleared = await page.evaluate(JS_CLEAR_SLICER, slicer_idx)
    if not cleared:
        print(f"    ⚠ Clear slicer failed — trying anyway")
    await page.wait_for_timeout(700)
    await page.evaluate(JS_SLICER_SCROLL_TOP, slicer_idx)
    await page.wait_for_timeout(200)
    for _ in range(40):
        if await page.evaluate(JS_TRY_SELECT_IN_SLICER, [slicer_idx, region]):
            return True
        r = await page.evaluate(JS_SCROLL_SLICER, [slicer_idx, 60])
        await page.wait_for_timeout(80)
        if r.get("done"):
            break
    await page.evaluate(JS_SLICER_SCROLL_TOP, slicer_idx)
    await page.wait_for_timeout(200)
    return await page.evaluate(JS_TRY_SELECT_IN_SLICER, [slicer_idx, region])

def build_region_slicers(fixed_template, region):
    result = []
    has_region = False
    for s in fixed_template:
        sc = dict(s)
        if norm(sc.get("title", "")) == norm("Region Depot"):
            sc["selected"]       = [region]
            sc["dax_filterable"] = True
            has_region = True
        result.append(sc)
    if not has_region:
        result.append({"title": "Region Depot", "selected": [region], "dax_filterable": True})
    return result

def build_tables_from_xlsx(fdf, table_meta_template):
    """Fallback: build national tables from xlsx when DOM slicer fails."""
    tables = []
    de_col   = next((c for c in fdf.columns if norm(c) == norm("Dernier E")), None)
    crbt_col = next((c for c in fdf.columns if norm(c) == "crbt"), None)

    for meta in table_meta_template:
        headers  = meta["headers"]
        dim_idx  = find_dim_col(headers)
        if dim_idx is None:
            tables.append({"title": meta.get("title",""), "headers": headers,
                           "rows": [], "num_rows": 0, "num_cols": len(headers)})
            continue
        dim_col_name  = headers[dim_idx]
        xlsx_dim      = next((c for c in fdf.columns if norm(c) == norm(dim_col_name)), None)
        if not xlsx_dim:
            tables.append({"title": meta.get("title",""), "headers": headers,
                           "rows": [], "num_rows": 0, "num_cols": len(headers)})
            continue

        non_dim_hdrs  = [h for i, h in enumerate(headers)
                         if i != dim_idx and "total" not in h.lower()]
        is_categorical = de_col and any(h in set(fdf[de_col].str.strip().unique())
                                         for h in non_dim_hdrs)
        dim_vals   = sorted([v for v in fdf[xlsx_dim].unique() if str(v).strip()])
        col_totals = [0] * len(headers)
        rows_out   = []
        for dv in dim_vals:
            sub = fdf[fdf[xlsx_dim] == dv]
            row = [''] * len(headers)
            row[dim_idx] = str(dv)
            for ci, h in enumerate(headers):
                if ci == dim_idx: continue
                h_norm = h.lower()
                if "total" in h_norm:
                    cnt = len(sub); row[ci] = str(cnt) if cnt else ''; col_totals[ci] += cnt
                elif is_categorical and de_col and h in set(fdf[de_col].str.strip().unique()):
                    cnt = int((sub[de_col].str.strip() == h).sum()); row[ci] = str(cnt) if cnt else ''; col_totals[ci] += cnt
                elif crbt_col and h_norm in ("crbt",):
                    vals = pd.to_numeric(sub[crbt_col], errors="coerce").fillna(0)
                    cnt  = int((vals > 0).sum()); row[ci] = str(cnt) if cnt else ''; col_totals[ci] += cnt
                elif crbt_col and h_norm in ("ordinaire",):
                    vals = pd.to_numeric(sub[crbt_col], errors="coerce").fillna(0)
                    cnt  = int((vals == 0).sum()); row[ci] = str(cnt) if cnt else ''; col_totals[ci] += cnt
            rows_out.append(row)
        tot_row = [''] * len(headers)
        for ci, tot in enumerate(col_totals):
            if tot: tot_row[ci] = str(tot)
        rows_out.append(tot_row)
        tables.append({"title": meta.get("title",""), "headers": headers,
                       "rows": rows_out, "num_rows": len(rows_out), "num_cols": len(headers)})
    return tables

# ── Export & Import table builders ───────────────────────────────────────────

def build_depot_export_table(export_df, region):
    """Table DÉPÔT EXPORT: Bureau depot × CRBT|Ordinaire|Total (from export sheet)."""
    headers = ["Bureau depot", "CRBT", "Ordinaire", "Total"]
    fdf = export_df[export_df["Region Depot"].str.strip() == region].copy()
    if fdf.empty:
        return {"title": "Export", "headers": headers,
                "rows": [], "num_rows": 0, "num_cols": len(headers)}
    bureaux     = sorted(fdf["Bureau depot"].str.strip().unique())
    rows        = []
    tot_crbt = tot_ord = tot_all = 0
    for b in bureaux:
        sub    = fdf[fdf["Bureau depot"].str.strip() == b]
        crbt_n = int((sub["CRBT/ORD"].str.strip() == "CRBT").sum())
        ord_n  = int((sub["CRBT/ORD"].str.strip() == "Ordinaire").sum())
        tot    = crbt_n + ord_n
        tot_crbt += crbt_n; tot_ord += ord_n; tot_all += tot
        rows.append([b,
                     str(crbt_n) if crbt_n else "",
                     str(ord_n)  if ord_n  else "",
                     str(tot)    if tot    else ""])
    rows.append(["", str(tot_crbt), str(tot_ord), str(tot_all)])
    return {"title": "Export", "headers": headers,
            "rows": rows, "num_rows": len(rows), "num_cols": len(headers)}

def build_livraison_import_table(import_df, region, events):
    """Table LIVRAISON IMPORT: Bureau dernier E × events|Total (from import.xls)."""
    headers = ["Bureau dernier E"] + events + ["Total"]
    fdf = import_df[import_df["Region dernier E"].str.strip() == region].copy()
    if fdf.empty:
        return {"title": "Import", "headers": headers,
                "rows": [], "num_rows": 0, "num_cols": len(headers)}
    bureaux    = sorted(fdf["Bureau dernier E"].str.strip().unique())
    rows       = []
    col_totals = [0] * len(headers)
    for b in bureaux:
        sub = fdf[fdf["Bureau dernier E"].str.strip() == b]
        row = [b]
        for j, evt in enumerate(events):
            cnt = int((sub["Dernier E"].str.strip() == evt).sum())
            row.append(str(cnt) if cnt else "")
            col_totals[j + 1] += cnt
        tot = len(sub)
        row.append(str(tot))
        col_totals[-1] += tot
        rows.append(row)
    tot_row = ([""] +
               [str(col_totals[j+1]) if col_totals[j+1] else "" for j in range(len(events))] +
               [str(col_totals[-1])])
    rows.append(tot_row)
    return {"title": "Import", "headers": headers,
            "rows": rows, "num_rows": len(rows), "num_cols": len(headers)}

# ── Drill mapping builders for xlsx-based tables ──────────────────────────────

def build_export_drill(export_df, region, tbl_key):
    """Drill mapping for export table: Bureau depot → MAILITM_FIDs with CA/CRBT split."""
    fdf = export_df[export_df["Region Depot"].str.strip() == region]
    flat_map = {}
    for _, row in fdf.iterrows():
        bureau = str(row.get("Bureau depot", "")).strip()
        id_    = str(row.get("MAILITM_FID", "")).strip()
        if not (bureau and id_): continue
        crbt_ord = str(row.get("CRBT/ORD", "")).strip()
        ca_v     = str(row.get("CA", "")).strip()
        entry    = {"id": id_, "poids": str(row.get("poids", "")).strip(), "CA": ca_v}
        bkt = flat_map.setdefault(bureau, {"__all__": []})
        bkt["__all__"].append(entry)
        if crbt_ord in ("CRBT", "Ordinaire"):
            bkt.setdefault(crbt_ord, []).append(entry)
    return {tbl_key: {"cols": ["poids", "CA"], "dim_idx": 0,
                      "headers": ["Bureau depot", "CRBT", "Ordinaire", "Total"],
                      "data": flat_map}}

def build_import_drill(import_df, region, events, tbl_key):
    """Drill mapping for import table: Bureau dernier E → MAILITM_FIDs per event."""
    fdf = import_df[import_df["Region dernier E"].str.strip() == region]
    flat_map = {}
    for _, row in fdf.iterrows():
        bureau = str(row.get("Bureau dernier E", "")).strip()
        id_    = str(row.get("MAILITM_FID", "")).strip()
        if not (bureau and id_): continue
        evt   = str(row.get("Dernier E", "")).strip()
        entry = {"id": id_, "poids": str(row.get("poids", "")).strip()}
        bkt   = flat_map.setdefault(bureau, {"__all__": []})
        bkt["__all__"].append(entry)
        if evt:
            bkt.setdefault(evt, []).append(entry)
    headers = ["Bureau dernier E"] + events + ["Total"]
    return {tbl_key: {"cols": ["poids"], "dim_idx": 0,
                      "headers": headers, "data": flat_map}}

# ── Email sender ──────────────────────────────────────────────────────────────

def send_region_email(filename, html_filename, slicers, tables,
                      timestamp, recipient, region, logo_b64, has_logo,
                      kpis=None, section_labels=None):
    email_html = build_email_html(filename, slicers, tables, timestamp,
                                  has_logo, logo_b64, kpis, section_labels)
    msg = MIMEMultipart("related")
    msg["From"]    = EMAIL_FROM
    msg["To"]      = recipient
    msg["Subject"] = (
        f"[La Poste Tunisienne] Rapport National — "
        f"{region} — "
        f"{datetime.strptime(timestamp, '%Y-%m-%d %H:%M').strftime('%d/%m/%Y')}"
    )
    msg.attach(MIMEText(email_html, "html", "utf-8"))

    with open(filename, "rb") as f:
        img = MIMEImage(f.read())
    img.add_header("Content-ID", "<report_img>")
    img.add_header("Content-Disposition", "inline", filename=filename)
    msg.attach(img)

    with open(html_filename, "rb") as f:
        att = MIMEBase("text", "html")
        att.set_payload(f.read())
    encoders.encode_base64(att)
    att.add_header("Content-Disposition", "attachment",
                   filename=os.path.basename(html_filename))
    msg.attach(att)

    with smtplib.SMTP_SSL("smtp.gmail.com", 465) as server:
        server.login(EMAIL_FROM, EMAIL_PASSWORD)
        server.sendmail(EMAIL_FROM, [recipient], msg.as_string())

# ── Main ──────────────────────────────────────────────────────────────────────

async def run_all_regions():
    has_logo = os.path.exists(LOGO_PATH)
    logo_b64 = get_logo_b64()

    # ── Load all 3 data sources ───────────────────────────────────────────────
    try:
        df = load_xlsx_df()
        print(f"✅ National xlsx chargé : {len(df)} lignes")
    except Exception as e:
        print(f"❌ National xlsx : {e}"); return

    try:
        export_df = pd.read_excel(XLSX_PATH, sheet_name="export", header=2, dtype=str).fillna("")
        print(f"✅ Export xlsx chargé  : {len(export_df)} lignes")
    except Exception as e:
        print(f"❌ Export xlsx : {e}"); export_df = None

    try:
        import_df = pd.read_excel(IMPORT_XLS_PATH, dtype=str).fillna("")
        import_events = sorted(set(
            e.strip() for e in import_df["Dernier E"].dropna().unique() if e.strip()
        ))
        print(f"✅ Import xls chargé   : {len(import_df)} lignes, {len(import_events)} événements")
    except Exception as e:
        print(f"❌ Import xls : {e}"); import_df = None; import_events = []

    async with async_playwright() as p:
        context = await p.chromium.launch_persistent_context(
            user_data_dir=PROFILE_DIR, headless=True,
            viewport={"width": 1920, "height": 1080}, device_scale_factor=2,
        )
        page = await context.new_page()

        token_holder = []
        def on_request(request):
            auth = request.headers.get("authorization", "")
            if auth.startswith("Bearer ") and not token_holder:
                token_holder.append(auth[7:])
        page.on("request", on_request)

        await page.goto(REPORT_URL, timeout=60000)
        try:
            await page.wait_for_load_state("load", timeout=15000)
        except Exception:
            pass

        if "app.powerbi.com" not in page.url:
            print("❌ Session expirée — relancez Step 1.")
            await context.close(); return

        await page.wait_for_timeout(8000)
        bearer_token = token_holder[0] if token_holder else None
        print(f"  {'✅ Bearer token capturé' if bearer_token else '⚠ Bearer token absent'}")

        dataset_id = get_dataset_id(bearer_token) if bearer_token else None
        tbl_name   = find_pbi_table(bearer_token, dataset_id) if dataset_id else None
        print(f"  ↳ Dataset : {dataset_id}\n  ↳ Table   : {tbl_name}")

        current_slicers = await page.evaluate(JS_SLICERS_GLOBAL)
        print(f"  Slicers bruts : {current_slicers}")
        has_agences = any(FIXED_CATEGORY in s.get("selected", []) for s in current_slicers)
        if not has_agences:
            print(f"❌ '{FIXED_CATEGORY}' n'est pas coché dans Power BI.")
            print(f"   → Sélectionnez-le manuellement puis relancez Step 3.")
            await context.close(); return
        print(f"  ✅ Filtre '{FIXED_CATEGORY}' confirmé")

        fixed_slicers_template = fix_slicer_titles(
            current_slicers, df, bearer_token, dataset_id, tbl_name)
        print(f"  ↳ Slicers résolus : {[(s['title'], s['selected'], s.get('dax_filterable')) for s in fixed_slicers_template]}")

        containers_info = await page.evaluate(JS_DUMP_CONTAINERS)
        print(f"  ↳ {len(containers_info)} visual-container(s) détectés")
        for ci in containers_info:
            print(f"    [{ci['idx']}] slicerTexts={ci['slicerTexts']} leafTexts={ci['leafTexts'][:3]}")

        slicer_idx    = await page.evaluate(JS_FIND_REGION_SLICER, _REGION_NAMES_UPPER)
        use_dom_slicer = slicer_idx >= 0
        if use_dom_slicer:
            print(f"  ✅ Slicer Region Depot trouvé (container #{slicer_idx})")
        else:
            print(f"  ⚠ Slicer Region Depot introuvable — mode xlsx fallback pour les tables nationaux")

        visuals_init   = await page.evaluate(JS_CLASSIFY)
        table_meta_tpl = []
        for v in visuals_init:
            if v["type"] != "table": continue
            hdrs = await page.evaluate(JS_HEADERS, v["index"])
            if not hdrs or is_mailitm_table(hdrs): continue
            table_meta_tpl.append({"key": f"t{len(table_meta_tpl)}",
                                   "idx": v["index"], "headers": hdrs,
                                   "title": v.get("title", "")})
        print(f"  ↳ {len(table_meta_tpl)} tableau(x) national template")

        success, failed = 0, []

        for region in _REGIONS:
            recipient = REGION_EMAILS.get(region, "").strip() or DEFAULT_EMAIL
            print(f"\n{'='*60}\n── {region} → {recipient} ──")

            slicers_reg = build_region_slicers(fixed_slicers_template, region)

            # ── KPIs from national xlsx ───────────────────────────────────────
            fdf = df.copy()
            for s in slicers_reg:
                if not s.get("title") or not s.get("selected"): continue
                col = next((c for c in fdf.columns if norm(c) == norm(s["title"])), None)
                if col:
                    fdf = xlsx_filter(fdf, col, s["selected"])
            kpis = compute_kpis(fdf)
            print(f"  ↳ KPIs : taux={kpis['taux']}%, total={kpis.get('total_ids', kpis['total'])}, CA={kpis.get('total_ca')}")

            # ── Extract national tables from PBI or build from xlsx ───────────
            if use_dom_slicer:
                ok = await set_region_slicer(page, slicer_idx, region)
                if not ok:
                    print(f"  ⚠ Slicer '{region}' non sélectionné — tables xlsx")
                    use_dom_for_this = False
                else:
                    print(f"  ✅ Slicer → {region}")
                    await page.wait_for_timeout(5000)
                    use_dom_for_this = True
            else:
                use_dom_for_this = False

            filename = f"report_{region}_{datetime.now().strftime('%Y%m%d_%H%M%S')}.png"
            await page.screenshot(path=filename, full_page=False)

            if use_dom_for_this:
                visuals = await page.evaluate(JS_CLASSIFY)
                nat_tables, nat_meta = [], []
                for v in visuals:
                    if v["type"] != "table": continue
                    headers = await page.evaluate(JS_HEADERS, v["index"])
                    if not headers or is_mailitm_table(headers): continue
                    rows = await extract_all_rows(page, v["index"], len(headers))
                    tkey = f"t{len(nat_tables)}"
                    nat_tables.append({"title": v["title"], "headers": headers,
                                       "rows": rows, "num_rows": len(rows), "num_cols": len(headers)})
                    nat_meta.append({"key": tkey, "idx": v["index"], "headers": headers})
                print(f"  ↳ {len(nat_tables)} tableau(x) national extraits (PBI)")
            else:
                nat_tables = build_tables_from_xlsx(fdf, table_meta_tpl)
                nat_meta   = [{"key": m["key"], "idx": m["idx"], "headers": m["headers"]}
                               for m in table_meta_tpl]
                print(f"  ↳ {len(nat_tables)} tableau(x) national construits (xlsx)")

            # ── Identify national depot (T1) and livraison (T2) ───────────────
            nat_t1 = next((t for t in nat_tables if is_depot_table(t["headers"])), None)
            nat_t2 = next((t for t in nat_tables if is_livraison_table(t["headers"])), None)
            nat_t1_meta = next((m for m in nat_meta if is_depot_table(m["headers"])), None)
            nat_t2_meta = next((m for m in nat_meta if is_livraison_table(m["headers"])), None)

            # ── Build national drill mappings ─────────────────────────────────
            nat_drill = build_drill_mappings(
                bearer_token, dataset_id, nat_tables, nat_meta, slicers_reg, df)

            # Identify which nat_drill key belongs to depot vs livraison
            nat_t1_drill_key, nat_t2_drill_key = None, None
            for key, dm in nat_drill.items():
                hdrs = dm.get("headers", []) if isinstance(dm, dict) else []
                if is_depot_table(hdrs):    nat_t1_drill_key = key
                elif is_livraison_table(hdrs): nat_t2_drill_key = key

            # ── Build export & import tables ──────────────────────────────────
            if export_df is not None:
                export_t = build_depot_export_table(export_df, region)
                export_drill = build_export_drill(export_df, region, "t1")
            else:
                headers_exp  = ["Bureau depot", "CRBT", "Ordinaire", "Total"]
                export_t     = {"title": "Export", "headers": headers_exp,
                                "rows": [], "num_rows": 0, "num_cols": len(headers_exp)}
                export_drill = {}

            if import_df is not None:
                import_t     = build_livraison_import_table(import_df, region, import_events)
                import_drill = build_import_drill(import_df, region, import_events, "t3")
            else:
                headers_imp  = ["Bureau dernier E", "Total"]
                import_t     = {"title": "Import", "headers": headers_imp,
                                "rows": [], "num_rows": 0, "num_cols": len(headers_imp)}
                import_drill = {}

            print(f"  ↳ Export : {export_t['num_rows']} lignes")
            print(f"  ↳ Import : {import_t['num_rows']} lignes")

            # ── Merge into final 4-table list with remapped drill keys ─────────
            # Order: [nat_t1(t0), export_t(t1), nat_t2(t2), import_t(t3)]
            all_tables = [
                nat_t1 or {"title": "National Dépôt", "headers": ["Bureau depot", "CRBT", "Ordinaire", "Total"],
                            "rows": [], "num_rows": 0, "num_cols": 4},
                export_t,
                nat_t2 or {"title": "National Livraison", "headers": ["Region dernier E", "Total"],
                            "rows": [], "num_rows": 0, "num_cols": 2},
                import_t,
            ]

            drill_mappings = {}
            if nat_t1_drill_key: drill_mappings["t0"] = nat_drill[nat_t1_drill_key]
            drill_mappings.update(export_drill)    # key "t1"
            if nat_t2_drill_key: drill_mappings["t2"] = nat_drill[nat_t2_drill_key]
            drill_mappings.update(import_drill)    # key "t3"

            for k, v in drill_mappings.items():
                data = v.get("data", {}) if isinstance(v, dict) else {}
                print(f"    [{k}] {len(data)} dim vals")

            # ── Write HTML ────────────────────────────────────────────────────
            timestamp     = datetime.now().strftime("%Y-%m-%d %H:%M")
            html_filename = filename.replace(".png", "_interactif.html")
            with open(html_filename, "w", encoding="utf-8") as hf:
                hf.write(build_interactive_html(
                    slicers_reg, all_tables, drill_mappings, timestamp,
                    logo_b64, kpis, section_labels=SECTION_LABELS))
            print(f"  ✅ HTML : {html_filename}")

            # ── Send email ────────────────────────────────────────────────────
            try:
                send_region_email(filename, html_filename, slicers_reg,
                                  all_tables, timestamp, recipient, region,
                                  logo_b64, has_logo, kpis,
                                  section_labels=SECTION_LABELS)
                print(f"  Email envoyé → {recipient}")
                success += 1
            except Exception as e:
                print(f"  ❌ Échec envoi : {e}")
                failed.append(region)

        await context.close()

    print(f"\n{'='*60}")
    print(f"✅ {success}/{len(_REGIONS)} emails envoyés")
    if failed:
        print(f"❌ Échecs : {failed}")

def run_regions():
    loop = asyncio.ProactorEventLoop() if sys.platform == "win32" else asyncio.new_event_loop()
    asyncio.set_event_loop(loop)
    try:
        loop.run_until_complete(run_all_regions())
    finally:
        loop.close()

thread = threading.Thread(target=run_regions)
thread.start()
thread.join()


  ↳ Categorie_Bureau_Dernier_E_nle computed from [Bureau dernier E]
✅ National xlsx chargé : 24003 lignes
✅ Export xlsx chargé  : 16417 lignes
✅ Import xls chargé   : 5466 lignes, 30 événements
  ✅ Bearer token capturé
  ↳ Dataset ID: 96786199-742d-4006-b6c5-acf1ce393a8a
  ↳ Dataset : 96786199-742d-4006-b6c5-acf1ce393a8a
  ↳ Table   : None
  Slicers bruts : []
❌ 'Agences' n'est pas coché dans Power BI.
   → Sélectionnez-le manuellement puis relancez Step 3.


In [ ]:
# ── STEP 4 (NATIONAL): Watch mode — régénération automatique ─────────────────
# Ouvre le rapport en mode visible. Chaque fois que vous changez un slicer,
# le fichier "national_live.html" est régénéré automatiquement.
# Ouvrez ce fichier dans votre navigateur (il se rafraîchit toutes les 6s).
# Arrêtez avec le bouton Stop du kernel Jupyter.

import asyncio, sys, threading
from playwright.async_api import async_playwright

LIVE_HTML = "national_live.html"
HEADLESS  = False   # True = invisible, False = vous voyez le navigateur

async def watch_loop():
    last_state = None

    async with async_playwright() as p:
        context = await p.chromium.launch_persistent_context(
            user_data_dir=PROFILE_DIR,
            headless=HEADLESS,
            viewport={"width": 1920, "height": 1080},
        )
        page = await context.new_page()

        token_holder = []
        def on_req(req):
            auth = req.headers.get("authorization", "")
            if auth.startswith("Bearer ") and not token_holder:
                token_holder.append(auth[7:])
        page.on("request", on_req)

        await page.goto(REPORT_URL, timeout=60000)
        try: await page.wait_for_load_state("load", timeout=15000)
        except: pass

        if "app.powerbi.com" not in page.url:
            print("Session expirée — relancez Step 1."); await context.close(); return

        await page.wait_for_timeout(8000)
        bearer_token = token_holder[0] if token_holder else None

        try:
            df = load_xlsx_df()
        except Exception as e:
            print(f"Erreur xlsx: {e}"); await context.close(); return

        dataset_id = get_dataset_id(bearer_token) if bearer_token else None

        print(f"Watch mode actif — ouvrez {LIVE_HTML} dans votre navigateur")
        print("Modifiez les slicers dans Power BI pour régénérer.")
        print("Arrêtez avec Stop (kernel Jupyter)\n")

        logo_b64 = get_logo_b64()

        while True:
            try:
                slicers = await page.evaluate(JS_SLICERS_GLOBAL)
                state   = str(sorted(
                    (s.get("title",""), tuple(sorted(s.get("selected",[]))))
                    for s in slicers))

                if state == last_state:
                    await asyncio.sleep(3)
                    continue

                if last_state is not None:
                    print(f"\n🔄 Changement détecté — attente mise à jour Power BI...")
                    await asyncio.sleep(5)

                last_state = state
                ts = __import__("datetime").datetime.now().strftime("%Y-%m-%d %H:%M")

                # Extract tables
                visuals = await page.evaluate(JS_CLASSIFY)
                tables, table_meta = [], []
                for v in visuals:
                    if v["type"] != "table": continue
                    headers = await page.evaluate(JS_HEADERS, v["index"])
                    if not headers or is_mailitm_table(headers): continue
                    rows = await extract_all_rows(page, v["index"], len(headers))
                    tkey = f"t{len(tables)}"
                    tables.append({"title": v["title"], "headers": headers,
                                   "rows": rows, "num_rows": len(rows), "num_cols": len(headers)})
                    table_meta.append({"key": tkey, "idx": v["index"], "headers": headers})

                # KPIs
                fdf = df.copy()
                slicers_copy = [dict(s) for s in slicers]
                slicers_copy = fix_slicer_titles(slicers_copy, df, bearer_token, dataset_id,
                                                 find_pbi_table(bearer_token, dataset_id) if bearer_token and dataset_id else None)
                for s in slicers_copy:
                    if not s.get("title") or not s.get("selected"): continue
                    col2 = next((c for c in fdf.columns if norm(c) == norm(s["title"])), None)
                    if col2:
                        fdf = xlsx_filter(fdf, col2, s["selected"])
                kpis = compute_kpis(fdf)

                # Drill mappings
                drill_mappings = build_drill_mappings(
                    bearer_token, dataset_id, tables, table_meta, slicers, df)

                # Write HTML with auto-refresh
                with open(LIVE_HTML, "w", encoding="utf-8") as f:
                    f.write(build_interactive_html(
                        slicers, tables, drill_mappings, ts, logo_b64,
                        kpis=kpis, auto_refresh=True))

                pct = kpis.get("taux", 0)
                avg = kpis.get("avg_intervalle", "?")
                print(f"  ✅ {ts} — HTML mis à jour | livraison: {pct}% | délai moy: {avg}j")

            except asyncio.CancelledError:
                break
            except Exception as e:
                print(f"  ⚠ {e}")
                await asyncio.sleep(3)

        await context.close()

def run_watch():
    loop = asyncio.ProactorEventLoop() if sys.platform == "win32" else asyncio.new_event_loop()
    asyncio.set_event_loop(loop)
    try:
        loop.run_until_complete(watch_loop())
    except KeyboardInterrupt:
        pass
    finally:
        loop.close()

threading.Thread(target=run_watch, daemon=True).start()


Exception in thread Thread-19 (run_watch):
Traceback (most recent call last):
  File "c:\Users\USER\miniconda3\Lib\threading.py", line 1073, in _bootstrap_inner
    self.run()
  File "c:\Users\USER\miniconda3\Lib\threading.py", line 1010, in run
    self._target(*self._args, **self._kwargs)
  File "C:\Users\USER\AppData\Local\Temp\ipykernel_17368\571948504.py", line 123, in run_watch
  File "c:\Users\USER\miniconda3\Lib\asyncio\base_events.py", line 687, in run_until_complete
    return future.result()
           ^^^^^^^^^^^^^^^
  File "C:\Users\USER\AppData\Local\Temp\ipykernel_17368\571948504.py", line 17, in watch_loop
  File "c:\Users\USER\miniconda3\Lib\site-packages\playwright\async_api\_generated.py", line 16606, in launch_persistent_context
    await self._impl_obj.launch_persistent_context(
  File "c:\Users\USER\miniconda3\Lib\site-packages\playwright\_impl\_browser_type.py", line 166, in launch_persistent_context
    result = await self._channel.send_return_as_dict(
         

  ⚠ Page.evaluate: Target page, context or browser has been closed
  ⚠ Page.evaluate: Target page, context or browser has been closed
  ⚠ Page.evaluate: Target page, context or browser has been closed
  ⚠ Page.evaluate: Target page, context or browser has been closed
  ⚠ Page.evaluate: Target page, context or browser has been closed
  ⚠ Page.evaluate: Target page, context or browser has been closed
  ⚠ Page.evaluate: Target page, context or browser has been closed
  ⚠ Page.evaluate: Target page, context or browser has been closed
  ⚠ Page.evaluate: Target page, context or browser has been closed
  ⚠ Page.evaluate: Target page, context or browser has been closed
  ⚠ Page.evaluate: Target page, context or browser has been closed
  ⚠ Page.evaluate: Target page, context or browser has been closed
  ⚠ Page.evaluate: Target page, context or browser has been closed
  ⚠ Page.evaluate: Target page, context or browser has been closed
  ⚠ Page.evaluate: Target page, context or browser has been cl

In [8]:
# ── DIAGNOSTIC : Tables et slicers du rapport National ──────────────────────
# Exécutez après Step 1 pour voir exactement quelles colonnes Power BI affiche.

import asyncio, sys, threading
from playwright.async_api import async_playwright

async def diagnose_national():
    async with async_playwright() as p:
        context = await p.chromium.launch_persistent_context(
            user_data_dir=PROFILE_DIR, headless=True,
            viewport={"width": 1920, "height": 1080},
        )
        page = await context.new_page()
        await page.goto(REPORT_URL, timeout=60000)
        try:
            await page.wait_for_load_state("load", timeout=15000)
        except Exception:
            pass
        if "app.powerbi.com" not in page.url:
            print("Session expirée — relancez Step 1."); await context.close(); return
        await page.wait_for_timeout(8000)

        print(f"URL actuelle : {page.url}")
        try:
            await page.wait_for_selector('[data-testid="visual-container"]', timeout=30000)
        except Exception as e:
            print(f"⚠ Aucun visual-container détecté après 30s : {e}")
        await page.screenshot(path="diag_debug.png", full_page=True)
        print("Screenshot debug : diag_debug.png")

        visuals = await page.evaluate(JS_CLASSIFY)
        print(f"Visuels détectés : {len(visuals)}")
        for v in visuals:
            print(f"  [{v['index']}] type={v['type']:5s}  titre='{v['title']}'")

        print("\nHeaders des tables :")
        for v in visuals:
            if v['type'] != 'table': continue
            headers = await page.evaluate(JS_HEADERS, v['index'])
            print(f"  [{v['index']}] '{v['title']}' → {headers}")

        print("\nPremières lignes de chaque table :")
        for v in visuals:
            if v['type'] != 'table': continue
            headers = await page.evaluate(JS_HEADERS, v['index'])
            if not headers: continue
            rows = await page.evaluate(JS_VISIBLE_ROWS, [v['index'], len(headers)])
            print(f"  [{v['index']}] '{v['title']}' — {len(rows)} ligne(s) visibles :")
            for r in rows[:4]:
                print(f"    {r}")

        print("\nSlicers détectés :")
        slicers = await page.evaluate(JS_SLICERS_GLOBAL)
        if slicers:
            for s in slicers:
                print(f"  titre='{s['title']}'  valeurs={s['selected']}")
        else:
            print("  (aucun slicer coché)")

        await context.close()

def run_diag():
    loop = asyncio.ProactorEventLoop() if sys.platform == 'win32' else asyncio.new_event_loop()
    asyncio.set_event_loop(loop)
    try: loop.run_until_complete(diagnose_national())
    finally: loop.close()

threading.Thread(target=run_diag).start()
